# SepsisGuard-CP — analysis notebook

This notebook reproduces every result of the article *Transporting early sepsis warnings between hospitals: a locked-protocol, bidirectional evaluation of calibration, conformal coverage, alarm burden and local adaptation*.

**Data.** PhysioNet/Computing in Cardiology Challenge 2019, version 1.0.0 (Reyna et al., Crit Care Med 2020;48:210–217), CC BY 4.0, https://doi.org/10.13026/v64v-d857. Hourly records of adult ICU stays from two hospital systems: A (20,336 stays) and B (20,000 stays). The Challenge label is positive from 6 h before the Sepsis-3 onset time t_sepsis. The data are downloaded from the PhysioNet open-data mirror in Cell 3.

**Protocol (SepsisGuard-CP).** Past-only temporal features → candidate models → primary model chosen by a pre-specified simplicity rule on VA_FIT → Platt recalibration on VA_FIT → Mondrian and class-conditional patient-level split conformal prediction on VA_CONF → alarm threshold set by a false-alarm budget (≤ 0.20 episodes per non-septic patient-day) → evaluation on the internal test set and, unchanged, at the other hospital → local recalibration and retraining arms.

**Run modes (Cell 1).**

| Mode | Patients A / B | Approx. runtime (Colab CPU) | Use |
|---|---|---|---|
| SMOKE_TEST | 1,500 / 800 | 10–15 min | check that every cell runs |
| STANDARD_RUN | 8,000 / 5,000 | about 1 h | quick exploration |
| FULL_PAPER_RUN | 20,336 / 20,000; 3 seeds; 1,000 bootstrap resamples | 3–5 h on first run; minutes when resumed | reproduces the reported results |

Every stage is checkpointed on Google Drive, so an interrupted session resumes from the last finished step. Cell 25 can be run on its own to list finished stages and zip all results saved so far.

**Cell map.**

| Cells | Content | Article |
|---|---|---|
| 1–4 | Configuration, data download, loading and audit | Section 3.1, Table 2 |
| 5 | Past-only feature construction with equivalence and causality checks | Section 3.2, Table S1 |
| 6–8c | Metrics, partitions, candidate models, clinical scores, hyperparameter search | Sections 3.3–3.4, Tables S2, S12 |
| 9–13 | Calibration, conformal prediction, alarm events, cross-site recalibration and label budget (A → B development configuration) | Sections 3.5–3.7, Tables S11, S14, S15 |
| 14–21d | Ablations, robustness, subgroups, explainability, GRU, bootstrap statistics, seeds, efficiency, time-feature check, label-free weighting, α sensitivity | Section 4.7, Table 6, Tables S4–S8, S13, S16, S17 |
| 21e | Locked analysis protocol (SHA-256 fingerprint) | Section 3.9 |
| 21f–21g | Bidirectional analysis A → B and B → A, locked hypotheses H1–H5 | Sections 4.1–4.6, Tables 3–5, Figs 2–6 |
| 21h | Sensitivity analyses added after the locked analysis (equal-burden comparison, budget robustness) | Sections 4.4–4.5, Table S9 |
| 22 | Figures | — |
| 23–24 | Integrity checks, run summary, model card, export | — |
| 25 | Stand-alone recovery cell | — |

Do not edit Cell 21e. On its first run it writes `config/PROTOCOL_v9_LOCK.json`; later runs stop if the protocol text no longer matches the stored fingerprint.

This notebook evaluates retrospective early-warning performance. It does not establish clinical efficacy or deployment safety.


In [ ]:
# Cell 1 — Configuration: choose exactly ONE run mode
import os
RUN_MODE = os.environ.get("SEPSISGUARD_MODE", "FULL_PAPER_RUN")   # "SMOKE_TEST" | "STANDARD_RUN" | "FULL_PAPER_RUN"

MODES = {
    "SMOKE_TEST": dict(max_pat_A=1500, max_pat_B=800, seeds=[42], boot=50,
                       gru_epochs=2, gru_max_pat=800, shap_rows=500, perm_rows=3000,
                       robust_pat=300, budget_reps=3, hpo_trials=3, full_cohort=False),
    "STANDARD_RUN": dict(max_pat_A=8000, max_pat_B=5000, seeds=[42], boot=200,
                         gru_epochs=8, gru_max_pat=4000, shap_rows=2000, perm_rows=15000,
                         robust_pat=1000, budget_reps=10, hpo_trials=6, full_cohort=False),
    "FULL_PAPER_RUN": dict(max_pat_A=None, max_pat_B=None, seeds=[42, 1234, 2025], boot=1000,
                           gru_epochs=15, gru_max_pat=None, shap_rows=3000, perm_rows=30000,
                           robust_pat=None, budget_reps=30, hpo_trials=8, full_cohort=True),
}
assert RUN_MODE in MODES, f"RUN_MODE must be one of {list(MODES)}"
CFG = dict(MODES[RUN_MODE])
SEED = CFG["seeds"][0]

ALPHA = 0.10                      # conformal miscoverage (90% nominal)
WINDOWS = (6, 12, 24)             # past-only rolling windows (hours)
SPLIT_FRACS = (0.60, 0.10, 0.10, 0.20)   # site A patients: TRAIN / VA_FIT / VA_CONF / TEST
B_CAL_FRAC = 0.20                 # site B patients reserved for recalibration experiments only
PRIMARY_CALIBRATOR = "platt"      # pre-declared; "temperature" and "isotonic" are reported as comparators
BUDGETS = [25, 50, 100, 200, 400, 800, 1600]   # local-patient budgets for the recalibration curve
GRU_WINDOW = 12
# Challenge utility parameters (Reyna et al. 2020, official evaluation code)
DT_EARLY, DT_OPTIMAL, DT_LATE = -12, -6, 3
MAX_U_TP, MIN_U_FN, U_FP, U_TN = 1.0, -2.0, -0.05, 0.0

print(f"RUN_MODE = {RUN_MODE} | full cohort = {CFG['full_cohort']}")
print(f"Seeds = {CFG['seeds']} | bootstrap reps = {CFG['boot']} | patients A/B cap = {CFG['max_pat_A']}/{CFG['max_pat_B']}")
if not CFG["full_cohort"]:
    print("Note: this mode uses a subset of patients and fewer resamples; use FULL_PAPER_RUN to reproduce the reported results.")


In [ ]:
# Cell 2 — Setup: imports, optional installs, versions, seeds, output folders
import os, sys, time, json, math, gc, hashlib, platform, subprocess, warnings, zipfile, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import minimize_scalar
import sklearn
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
import joblib

def _ensure(module, pip_name=None):
    """Import a module, installing a binary wheel once if it is missing. Returns None on failure."""
    try:
        return __import__(module)
    except ImportError:
        name = pip_name or module
        print(f"[setup] installing {name} ...")
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", name], capture_output=True, text=True)
        if r.returncode != 0:
            print(f"[setup] WARNING: could not install {name}: {r.stderr[-300:]}")
            return None
        return __import__(module)

lgb = _ensure("lightgbm")
xgb = _ensure("xgboost")
shap = _ensure("shap")
try:
    import torch
    import torch.nn as nn
except ImportError:
    torch = None
HAVE_LGB, HAVE_XGB, HAVE_SHAP, HAVE_TORCH = lgb is not None, xgb is not None, shap is not None, torch is not None
if HAVE_LGB:
    from lightgbm import LGBMClassifier
if HAVE_XGB:
    from xgboost import XGBClassifier

def set_seeds(s):
    random.seed(s); np.random.seed(s)
    if HAVE_TORCH:
        torch.manual_seed(s)
set_seeds(SEED)

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "scipy": scipy.__version__, "matplotlib": matplotlib.__version__,
            "lightgbm": getattr(lgb, "__version__", None), "xgboost": getattr(xgb, "__version__", None),
            "shap": getattr(shap, "__version__", None), "torch": getattr(torch, "__version__", None)}
RUNTIME = {"platform": platform.platform(), "cpu_count": os.cpu_count(),
           "gpu": bool(HAVE_TORCH and torch.cuda.is_available())}
print(json.dumps({**VERSIONS, **RUNTIME}, indent=1))

# Drive (persists across disconnects); falls back to the working directory outside Colab
IN_COLAB = "google.colab" in sys.modules
DRIVE = None
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE = Path("/content/drive/MyDrive")
    except Exception as e:
        print(f"[setup] Drive unavailable ({e}). Outputs will be written to /content and will NOT survive a disconnect.")
BASE = Path(os.environ.get("SEPSISGUARD_BASE", str(DRIVE if DRIVE else (Path("/content") if IN_COLAB else Path.cwd()))))
DATA_ROOT = Path(os.environ.get("SEPSIS_DATA_ROOT", str(BASE / "sepsisguard_data")))
OUT = BASE / "SepsisGuard_CP_outputs" / RUN_MODE
SUB = {k: OUT / k for k in ["config", "data_audit", "checkpoints", "predictions", "tables", "figures", "models", "logs"]}
for p in [DATA_ROOT, *SUB.values()]:
    p.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT, "\nOUT:", OUT)

TABLES = {}          # name -> DataFrame, collected for the Excel export
LOG = []             # run log lines
def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    LOG.append(line); print(line)

def save_table(df, name, folder="tables"):
    path = SUB[folder] / f"{name}.csv"
    df.to_csv(path, index=False)
    TABLES[name] = df
    print(f"  saved {path.name}: shape={df.shape}, {path.stat().st_size/1024:.1f} KB")
    return path

OKABE_ITO = ["#000000", "#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7"]
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=OKABE_ITO), "font.size": 9})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(SUB["figures"] / f"{name}.{ext}", bbox_inches="tight")
    print(f"  saved figure {name}.png/.pdf")
    plt.close(fig)


# ---------------- Checkpointing and stage tracking (everything persists on Drive) ----------------
# A crash or disconnect late in a run must never lose finished work.
# 1) Every table is written to Drive the moment it is computed (save_table).
# 2) Every expensive step is checkpointed with an atomic write (tmp file + rename), so a disconnect
#    mid-write cannot leave a corrupted file. Re-running the notebook skips completed steps.
# 3) Optional stages (ablations, robustness, SHAP, GRU, statistics, figures) are isolated: if one fails,
#    its traceback goes to logs/ERROR_<stage>.txt and the run continues to the export cell, so all
#    finished results are still zipped.
import traceback
STATUS_PATH = SUB["logs"] / "run_status.json"
STATUS = json.load(open(STATUS_PATH)) if STATUS_PATH.exists() else {}
FAILED = {}
if STATUS:
    print("Resuming. Stages recorded by previous runs:")
    for k, v in STATUS.items():
        print(f"  {k:<28} {v.get('state'):<8} {v.get('time')}")

def mark(stage, state="DONE", **info):
    STATUS[stage] = {"state": state, "time": time.strftime("%Y-%m-%d %H:%M:%S"), **info}
    if state == "DONE":   # a stage that now succeeds clears its old error report
        FAILED.pop(stage, None)
        old_err = SUB["logs"] / f"ERROR_{stage}.txt"
        if old_err.exists(): old_err.unlink()
    tmp = STATUS_PATH.with_suffix(".tmp")
    with open(tmp, "w") as fh:
        json.dump(STATUS, fh, indent=1, default=str)
    os.replace(tmp, STATUS_PATH)

def atomic_dump(obj, path):
    path = Path(path); tmp = path.with_name(path.name + ".tmp")
    joblib.dump(obj, tmp); os.replace(tmp, path)

def ckpt_path(name):
    return SUB["checkpoints"] / f"{name}_{RUN_MODE}.joblib"

def cached(name, fn, key):
    """Return a checkpointed result if its key matches; otherwise compute, save atomically and return it."""
    path = ckpt_path(name)
    if path.exists():
        try:
            obj = joblib.load(path)
            if obj.get("key") == key:
                log(f"checkpoint hit: {name}")
                return obj["value"]
            log(f"checkpoint {name} has a different key (data/config changed); recomputing")
        except Exception as e:
            log(f"checkpoint {name} unreadable ({e}); recomputing")
    value = fn()
    atomic_dump({"key": key, "value": value}, path)
    log(f"checkpoint saved: {name}")
    return value

def stage_failed(stage, err):
    tb = traceback.format_exc()
    (SUB["logs"] / f"ERROR_{stage}.txt").write_text(tb)
    FAILED[stage] = f"{type(err).__name__}: {err}"
    mark(stage, "FAILED", error=FAILED[stage])
    print(f"[STAGE FAILED] {stage}: {FAILED[stage]}\n  traceback -> logs/ERROR_{stage}.txt; "
          "the run continues so finished results are kept.")

RUN_START = time.time()


In [ ]:
# Cell 3 — Dataset acquisition (official PhysioNet open-data S3 mirror, no credentials) + provenance
# Each hospital folder is synced directly into DATA_ROOT/A and DATA_ROOT/B. `aws s3 sync` is resumable:
# re-running this cell after a disconnect continues where it stopped.
EXPECTED_N = {"A": 20336, "B": 20000}
S3_BASE = "s3://physionet-open/challenge-2019/1.0.0/training/"
S3_FOLDERS = {"A": "training_setA", "B": "training_setB"}
SKIP_DOWNLOAD = os.environ.get("SEPSIS_SKIP_DOWNLOAD", "0") == "1"

def list_psv(site):
    d = DATA_ROOT / site
    return sorted(d.rglob("*.psv")) if d.exists() else []

# Re-use a v7 download that kept the bucket's folder names
for site, folder in S3_FOLDERS.items():
    legacy = DATA_ROOT / folder
    if legacy.exists() and not (DATA_ROOT / site).exists():
        legacy.rename(DATA_ROOT / site)
        log(f"renamed legacy folder {folder} -> {site}")

need = {s: len(list_psv(s)) < EXPECTED_N[s] for s in EXPECTED_N}
if any(need.values()) and not SKIP_DOWNLOAD:
    try:
        subprocess.run(["aws", "--version"], check=True, capture_output=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "awscli"], check=True)
    for site, missing in need.items():
        if missing:
            log(f"syncing hospital {site} from {S3_BASE}{S3_FOLDERS[site]}/ (resumable)")
            subprocess.run(["aws", "s3", "sync", "--no-sign-request", "--only-show-errors",
                            f"{S3_BASE}{S3_FOLDERS[site]}/", str(DATA_ROOT / site)], check=True)

filesA, filesB = list_psv("A"), list_psv("B")
log(f"files found: A={len(filesA)}  B={len(filesB)}")
if not filesA or not filesB:
    raise RuntimeError(
        "WHAT: PhysioNet 2019 .psv files not found.\n"
        f"WHY: the S3 sync did not complete, or files are not under {DATA_ROOT}/A and {DATA_ROOT}/B.\n"
        f"FIX: run `aws s3 sync --no-sign-request {S3_BASE}training_setA/ {DATA_ROOT}/A` (and setB -> B), "
        "or upload the files manually.")
if RUN_MODE == "FULL_PAPER_RUN" and (len(filesA) != EXPECTED_N["A"] or len(filesB) != EXPECTED_N["B"]):
    raise RuntimeError(f"FULL_PAPER_RUN needs the complete dataset (A={EXPECTED_N['A']}, B={EXPECTED_N['B']}); "
                       f"found A={len(filesA)}, B={len(filesB)}. Re-run this cell to resume the sync.")

CORE = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2", "BaseExcess", "HCO3", "FiO2", "pH", "PaCO2",
        "SaO2", "AST", "BUN", "Alkalinephos", "Calcium", "Chloride", "Creatinine", "Bilirubin_direct", "Glucose",
        "Lactate", "Magnesium", "Phosphate", "Potassium", "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT",
        "WBC", "Fibrinogen", "Platelets"]
VITALS = CORE[:8]
LABS = CORE[8:]
STATIC = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime"]
RAW_COLS = CORE + STATIC + ["ICULOS", "SepsisLabel"]

# Schema preflight on one real file per site
for f in (filesA[0], filesB[0]):
    hdr = pd.read_csv(f, sep="|", nrows=2)
    miss = [c for c in RAW_COLS if c not in hdr.columns]
    if miss:
        raise RuntimeError(f"Schema check failed for {f.name}: missing {miss}. The file may be HTML/empty or the release changed.")

DATA_PROVENANCE = {
    "name": "PhysioNet/Computing in Cardiology Challenge 2019: Early Prediction of Sepsis from Clinical Data",
    "version": "1.0.0", "doi": "10.13026/v64v-d857", "license": "CC BY 4.0",
    "url": "https://physionet.org/content/challenge-2019/1.0.0/", "source_mirror": S3_BASE,
    "citation": "Reyna MA et al. Crit Care Med 2020;48(2):210-217. doi:10.1097/CCM.0000000000004145",
    "files_A": len(filesA), "files_B": len(filesB), "format": "pipe-separated .psv, one file per ICU stay",
    "accessed": time.strftime("%Y-%m-%d"), "local_root": str(DATA_ROOT)}
with open(SUB["config"] / "DATA_PROVENANCE.json", "w") as fh:
    json.dump(DATA_PROVENANCE, fh, indent=2)
print(json.dumps(DATA_PROVENANCE, indent=1))
mark("03_data_acquisition")


In [ ]:
# Cell 4 — Load raw hourly records (parallel read, cached once to Drive) + data audit
# Files are read in parallel, the raw table is cached once, and every unreadable file is reported.
def read_psv(path, site):
    d = pd.read_csv(path, sep="|")
    miss = [c for c in RAW_COLS if c not in d.columns]
    if miss:
        raise ValueError(f"missing columns {miss}")
    d = d[RAW_COLS].copy()
    d.insert(0, "patient_id", f"{site}_{path.stem}")
    d.insert(1, "site", site)
    return d

def load_site_raw(files, site):
    cache = DATA_ROOT / "cache" / f"raw_{site}_v8.pkl"
    if cache.exists():
        df = pd.read_pickle(cache)
        if df["patient_id"].nunique() == len(files):
            log(f"raw cache hit for {site}: {len(df):,} rows")
            return df
        log(f"raw cache for {site} is stale ({df['patient_id'].nunique()} vs {len(files)} files); rebuilding")
    # Files are read in chunks of 2,000; each finished chunk is saved to Drive at once, so a disconnect
    # during the first (slow) read resumes from the last finished chunk instead of from zero.
    part_dir = DATA_ROOT / "cache" / f"parts_{site}"; part_dir.mkdir(parents=True, exist_ok=True)
    def job(p):
        try:
            return read_psv(p, site), None
        except Exception as e:
            return None, f"{p.name}: {e}"
    errors, parts, CH = [], [], 2000
    t0 = time.time()
    for k in range(0, len(files), CH):
        pf = part_dir / f"part_{k // CH:03d}_{len(files)}.pkl"
        if pf.exists():
            obj = pd.read_pickle(pf); parts.append(obj["df"]); errors += obj["errors"]; continue
        chunk_parts, chunk_err = [], []
        with ThreadPoolExecutor(max_workers=16) as ex:
            for d, e in ex.map(job, files[k:k + CH]):
                (chunk_err.append(e) if e else chunk_parts.append(d))
        cdf = pd.concat(chunk_parts, ignore_index=True) if chunk_parts else pd.DataFrame(columns=["patient_id", "site"] + RAW_COLS)
        tmp = pf.with_name(pf.name + ".tmp"); pd.to_pickle({"df": cdf, "errors": chunk_err}, tmp); os.replace(tmp, pf)
        parts.append(cdf); errors += chunk_err
        print(f"  {site}: {min(k + CH, len(files))}/{len(files)} files read and saved ({time.time()-t0:.0f}s)")
    if errors:
        pd.DataFrame({"error": errors}).to_csv(SUB["data_audit"] / f"read_errors_{site}.csv", index=False)
        print(f"  WARNING {site}: {len(errors)} unreadable files (listed in data_audit/read_errors_{site}.csv)")
        if len(errors) > 0.001 * len(files):
            raise RuntimeError(f"{len(errors)} files failed to parse for site {site}; inspect read_errors_{site}.csv")
    df = pd.concat(parts, ignore_index=True)
    for c in CORE + STATIC + ["ICULOS"]:
        df[c] = df[c].astype("float32")
    df["SepsisLabel"] = df["SepsisLabel"].astype("int8")
    cache.parent.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_name(cache.name + ".tmp"); df.to_pickle(tmp); os.replace(tmp, cache)
    log(f"raw {site}: {len(df):,} rows from {df['patient_id'].nunique():,} files cached to {cache}")
    return df

def subsample_patients(df, maxp, seed):
    if maxp is None:
        return df
    pats = np.sort(df["patient_id"].unique())
    keep = np.random.default_rng(seed).choice(pats, size=min(maxp, len(pats)), replace=False)
    return df[df["patient_id"].isin(set(keep))]

rawA = subsample_patients(load_site_raw(filesA, "A"), CFG["max_pat_A"], SEED)
rawB = subsample_patients(load_site_raw(filesB, "B"), CFG["max_pat_B"], SEED)
RAW = pd.concat([rawA, rawB], ignore_index=True).sort_values(["patient_id", "ICULOS"], kind="stable").reset_index(drop=True)
del rawA, rawB; gc.collect()

# ---------------- Data audit ----------------
same = RAW["patient_id"].to_numpy()[1:] == RAW["patient_id"].to_numpy()[:-1]
d_iculos = np.diff(RAW["ICULOS"].to_numpy())
d_label = np.diff(RAW["SepsisLabel"].to_numpy().astype(int))
audit_checks = {
    "duplicate (patient, ICULOS) rows": int(RAW.duplicated(["patient_id", "ICULOS"]).sum()),
    "non-increasing ICULOS within patient": int(np.sum(same & (d_iculos <= 0))),
    "ICULOS gaps > 1 h within patient": int(np.sum(same & (d_iculos > 1))),
    "label 1 -> 0 reversals within patient": int(np.sum(same & (d_label < 0))),
    "missing SepsisLabel": int(RAW["SepsisLabel"].isna().sum()),
}
RANGES = {"HR": (10, 300), "O2Sat": (10, 100), "Temp": (20, 45), "SBP": (20, 300), "MAP": (10, 300),
          "Resp": (1, 100), "pH": (6.5, 8.0), "Lactate": (0, 40), "WBC": (0, 500), "Platelets": (0, 3000)}
for c, (lo, hi) in RANGES.items():
    v = RAW[c].to_numpy()
    audit_checks[f"{c} outside [{lo},{hi}]"] = int(np.sum((v < lo) | (v > hi)))
audit_df = pd.DataFrame({"Check": list(audit_checks), "Count": list(audit_checks.values())})
assert audit_checks["duplicate (patient, ICULOS) rows"] == 0, "duplicate hourly rows found"
assert audit_checks["non-increasing ICULOS within patient"] == 0, "ICULOS not strictly increasing within patients"
assert audit_checks["missing SepsisLabel"] == 0

pat = RAW.groupby("patient_id", sort=True).agg(site=("site", "first"), rows=("ICULOS", "size"),
                                                septic=("SepsisLabel", "max"))
pat["septic"] = pat["septic"].astype(int)
cohort = pat.groupby("site").agg(patients=("rows", "size"), rows=("rows", "sum"),
                                 septic_patients=("septic", "sum"), median_LOS_h=("rows", "median")).reset_index()
cohort["septic_patient_pct"] = 100 * cohort["septic_patients"] / cohort["patients"]
row_prev = RAW.groupby("site")["SepsisLabel"].mean().mul(100).rename("positive_row_pct").reset_index()
cohort = cohort.merge(row_prev, on="site")
first_rows = RAW.drop_duplicates("patient_id")
stat_desc = first_rows.groupby("site").agg(age_median=("Age", "median"), male_frac=("Gender", "mean"),
                                           unit1_frac=("Unit1", "mean"), unit2_frac=("Unit2", "mean")).reset_index()
for a, b in [("male_frac", "male_pct"), ("unit1_frac", "unit1_MICU_pct"), ("unit2_frac", "unit2_SICU_pct")]:
    stat_desc[b] = 100 * stat_desc.pop(a)
cohort = cohort.merge(stat_desc, on="site")
missing_df = (RAW[CORE].isna().groupby(RAW["site"]).mean() * 100).T
missing_df.columns = [f"missing_pct_{c}" for c in missing_df.columns]
missing_df = missing_df.rename_axis("Variable").reset_index()

save_table(cohort, "T01_Cohort_Characteristics")
save_table(audit_df, "T02a_Data_Audit_Checks", "data_audit")
save_table(missing_df, "T02b_Missingness_By_Variable", "data_audit")
display(cohort); display(audit_df)

mark("04_raw_load_audit", rows=len(RAW))


In [ ]:
# Cell 5 — Vectorised, strictly past-only temporal features + correctness and causality assertions
# Every feature at ICU hour t uses rows <= t of the same patient only. No fitted transform is applied here;
# imputation and scaling are fitted later on TRAIN only.
FAMILY_OF = {}

def build_features(raw):
    """raw must be sorted by patient_id, ICULOS. Returns (X float32, names, meta dict)."""
    n = len(raw)
    pid = raw["patient_id"].to_numpy()
    newpat = np.r_[True, pid[1:] != pid[:-1]]
    ar = np.arange(n)
    start = np.maximum.accumulate(np.where(newpat, ar, 0))
    pcode = np.cumsum(newpat) - 1
    F = len(STATIC) + 1 + len(CORE) * (4 + 2 * len(WINDOWS)) + 5
    X = np.empty((n, F), dtype=np.float32)
    names = []
    def fill(name, arr, family):
        X[:, len(names)] = arr
        names.append(name); FAMILY_OF[name] = family
    for c in STATIC + ["ICULOS"]:
        fill(c, raw[c].to_numpy(np.float64), "static_context")
    meas24 = np.zeros(n)
    keep_last = {}
    for c in CORE:
        v = raw[c].to_numpy(np.float64)
        miss = np.isnan(v)
        fill(f"{c}_current", v, "current")
        fill(f"{c}_missing", miss.astype(np.float64), "missingness")
        last = pd.Series(v).groupby(pcode).ffill().to_numpy()
        fill(f"{c}_last", last, "last_observed")
        prev = np.empty(n); prev[0] = np.nan; prev[1:] = last[:-1]; prev[newpat] = np.nan
        fill(f"{c}_delta", last - prev, "delta")
        if c in ("HR", "SBP", "BUN", "Creatinine", "Resp", "Temp", "WBC"):
            keep_last[c] = last
        center = float(np.nanmedian(v)) if (~miss).any() else 0.0     # centring for numerically stable sums
        x0 = np.where(miss, 0.0, v - center)
        ok = (~miss).astype(np.float64)
        C1 = np.concatenate([[0.0], np.cumsum(x0)])
        C2 = np.concatenate([[0.0], np.cumsum(x0 * x0)])
        CN = np.concatenate([[0.0], np.cumsum(ok)])
        for w in WINDOWS:
            j = np.maximum(start, ar - w + 1)
            k = CN[ar + 1] - CN[j]
            s1 = C1[ar + 1] - C1[j]
            s2 = C2[ar + 1] - C2[j]
            mean = np.where(k > 0, s1 / np.maximum(k, 1) + center, np.nan)
            var = np.where(k > 1, (s2 - s1 * s1 / np.maximum(k, 1)) / np.maximum(k - 1, 1), np.nan)
            fam = "window_6h" if w == 6 else "window_12_24h"
            fill(f"{c}_mean{w}", mean, fam)
            fill(f"{c}_std{w}", np.sqrt(np.clip(var, 0, None)), fam)
            if w == 24:
                meas24 += k
    L = keep_last
    with np.errstate(divide="ignore", invalid="ignore"):
        fill("ShockIndex", np.where(L["SBP"] > 0, L["HR"] / L["SBP"], np.nan), "clinical_composite")
        fill("BUN_Creatinine", np.where(L["Creatinine"] > 0, L["BUN"] / L["Creatinine"], np.nan), "clinical_composite")
    fill("qSOFA_partial", (L["Resp"] >= 22).astype(float) + (L["SBP"] <= 100).astype(float), "clinical_composite")
    fill("SIRS_partial", ((L["Temp"] > 38) | (L["Temp"] < 36)).astype(float) + (L["HR"] > 90).astype(float)
         + (L["Resp"] > 20).astype(float) + ((L["WBC"] > 12) | (L["WBC"] < 4)).astype(float), "clinical_composite")
    fill("MeasCount24", meas24, "missingness")
    assert len(names) == F, (len(names), F)
    meta = dict(newpat=newpat, start=start, pcode=pcode, pos=ar - start)
    return X, names, meta

t0 = time.time()
X, FEATURES, META = build_features(RAW)
log(f"features built: X={X.shape} ({X.nbytes/1e9:.2f} GB float32) in {time.time()-t0:.1f}s")
assert "SepsisLabel" not in " ".join(FEATURES), "label leaked into feature names"

PID = RAW["patient_id"].to_numpy(); SITE = RAW["site"].to_numpy()
Y = RAW["SepsisLabel"].to_numpy().astype(np.int8)
POS, PCODE, START = META["pos"], META["pcode"], META["start"]
PAT_START = np.flatnonzero(META["newpat"])
N_PAT = len(PAT_START)
FEAT_HASH = hashlib.md5(("|".join(FEATURES) + f"{len(X)}").encode()).hexdigest()[:10]
FAMILIES = ["static_context", "current", "missingness", "last_observed", "window_6h", "window_12_24h", "delta", "clinical_composite"]
FAM_COLS = {f: [i for i, n in enumerate(FEATURES) if FAMILY_OF[n] == f] for f in FAMILIES}
save_table(pd.DataFrame({"Feature": FEATURES, "Family": [FAMILY_OF[n] for n in FEATURES]}), "T04_Feature_Dictionary", "config")

# (a) Equivalence with a naive reference implementation of the v7 definition (random real patients)
def reference_features(g):
    hist = {c: [] for c in CORE}
    rows = []
    for _, r in g.iterrows():
        z = {}
        for c in CORE:
            hist[c].append(r[c]); arr = pd.Series(hist[c], dtype=float)
            z[f"{c}_current"] = r[c]; z[f"{c}_missing"] = float(pd.isna(r[c]))
            z[f"{c}_last"] = arr.ffill().iloc[-1] if arr.notna().any() else np.nan
            for w in WINDOWS:
                q = arr.iloc[-w:]; z[f"{c}_mean{w}"] = q.mean(); z[f"{c}_std{w}"] = q.std()
            if len(arr) >= 2:
                prev = arr.iloc[:-1].ffill().iloc[-1] if arr.iloc[:-1].notna().any() else np.nan
                z[f"{c}_delta"] = z[f"{c}_last"] - prev if pd.notna(prev) and pd.notna(z[f"{c}_last"]) else np.nan
            else:
                z[f"{c}_delta"] = np.nan
        rows.append(z)
    return pd.DataFrame(rows)

chk_rng = np.random.default_rng(SEED)
check_pats = chk_rng.choice(N_PAT, size=min(8, N_PAT), replace=False)
col_idx = {n: i for i, n in enumerate(FEATURES)}
for pc in check_pats:
    rows = np.arange(PAT_START[pc], PAT_START[pc] + np.diff(np.r_[PAT_START, len(X)])[pc])
    ref = reference_features(RAW.iloc[rows])
    for name in ref.columns:
        a = X[rows, col_idx[name]].astype(np.float64); b = ref[name].to_numpy(np.float64)
        assert np.allclose(a, b, rtol=1e-4, atol=1e-3, equal_nan=True), f"feature mismatch {name} patient {PID[rows[0]]}"
log(f"feature equivalence check passed on {len(check_pats)} patients")

# (b) Causality: truncating a patient's future must not change any earlier feature value
PAT_ROWS = np.diff(np.r_[PAT_START, len(X)])
long_pats = np.flatnonzero(PAT_ROWS >= 30)[:3]
for pc in long_pats:
    rows = np.arange(PAT_START[pc], PAT_START[pc] + PAT_ROWS[pc])
    Xt, _, _ = build_features(RAW.iloc[rows[:15]].reset_index(drop=True))
    assert np.allclose(Xt, X[rows[:15]], equal_nan=True, rtol=1e-4, atol=1e-3), "future rows changed past features (leakage)"
log(f"causality (no look-ahead) check passed on {len(long_pats)} patients")
mark("05_features")


In [ ]:
# Cell 6 — Challenge utility (vectorised), metrics, calibration helpers + verification against a reference loop
BIG = 10**9
FIRST_LAB_P = np.minimum.reduceat(np.where(Y == 1, POS, BIG), PAT_START)
SEPTIC_P = FIRST_LAB_P < BIG
TSEP_P = np.where(SEPTIC_P, FIRST_LAB_P - DT_OPTIMAL, np.inf)     # t_sepsis = first positive label + 6 h (row index)
SEPTIC = SEPTIC_P[PCODE]; TSEP = TSEP_P[PCODE]

_m1 = MAX_U_TP / (DT_OPTIMAL - DT_EARLY); _b1 = -_m1 * DT_EARLY
_m2 = -MAX_U_TP / (DT_LATE - DT_OPTIMAL); _b2 = -_m2 * DT_LATE
_m3 = MIN_U_FN / (DT_LATE - DT_OPTIMAL); _b3 = -_m3 * DT_OPTIMAL

def utility_rows(pred, idx):
    """Per-row utility of boolean predictions `pred` (aligned with idx), following the Challenge definition."""
    pred = np.asarray(pred, bool)
    sep = SEPTIC[idx]; dt = POS[idx] - TSEP[idx]
    u = np.zeros(len(idx))
    u[~sep & pred] = U_FP
    u[~sep & ~pred] = U_TN
    inwin = sep & (dt <= DT_LATE)
    early = inwin & (dt <= DT_OPTIMAL); mid = inwin & (dt > DT_OPTIMAL)
    m = early & pred; u[m] = np.maximum(_m1 * dt[m] + _b1, U_FP)
    m = mid & pred;   u[m] = _m2 * dt[m] + _b2
    m = mid & ~pred;  u[m] = _m3 * dt[m] + _b3
    return u

ALL_IDX = np.arange(len(Y))
_best_pred = SEPTIC & (POS >= np.maximum(0, TSEP + DT_EARLY)) & (POS <= TSEP + DT_LATE)
U_BEST = utility_rows(_best_pred, ALL_IDX)
U_INACT = utility_rows(np.zeros(len(Y), bool), ALL_IDX)

def norm_utility(pred, idx):
    u = utility_rows(pred, idx)
    den = U_BEST[idx].sum() - U_INACT[idx].sum()
    return float((u.sum() - U_INACT[idx].sum()) / den) if den != 0 else np.nan

# Verification against a straightforward per-patient loop of the published utility definition
def _reference_patient_utility(labels, preds):
    septic = labels.any(); ts = int(np.argmax(labels)) - DT_OPTIMAL if septic else np.inf
    u = 0.0
    for t in range(len(labels)):
        if t <= ts + DT_LATE:
            if septic and preds[t]:
                u += max(_m1 * (t - ts) + _b1, U_FP) if t <= ts + DT_OPTIMAL else _m2 * (t - ts) + _b2
            elif not septic and preds[t]:
                u += U_FP
            elif septic and not preds[t]:
                u += 0 if t <= ts + DT_OPTIMAL else _m3 * (t - ts) + _b3
            else:
                u += U_TN
    return u
_r = np.random.default_rng(0)
_test_pats = np.r_[np.flatnonzero(SEPTIC_P)[:25], np.flatnonzero(~SEPTIC_P)[:25]]
for pc in _test_pats:
    rows = np.arange(PAT_START[pc], PAT_START[pc] + PAT_ROWS[pc])
    pr = _r.random(len(rows)) < 0.3
    assert abs(utility_rows(pr, rows).sum() - _reference_patient_utility(Y[rows] == 1, pr)) < 1e-9
log(f"utility implementation verified on {len(_test_pats)} patients")

def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7); return np.log(p / (1 - p))
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def ece_score(y, p, bins=15):
    b = np.minimum((p * bins).astype(int), bins - 1)
    sp = np.bincount(b, weights=p, minlength=bins); sy = np.bincount(b, weights=y, minlength=bins)
    return float(np.sum(np.abs(sy - sp)) / len(p))

def cal_slope(y, p):
    if len(np.unique(y)) < 2: return np.nan
    lr = LogisticRegression(C=1e6, max_iter=200).fit(logit(p).reshape(-1, 1), y)
    return float(lr.coef_[0, 0])

def evaluate(name, split, p, idx, thr):
    y = Y[idx].astype(int); pp = np.clip(np.asarray(p, float)[idx], 1e-7, 1 - 1e-7); pred = pp >= thr
    tp = int(np.sum(pred & (y == 1))); fp = int(np.sum(pred & (y == 0)))
    fn = int(np.sum(~pred & (y == 1))); tn = int(np.sum(~pred & (y == 0)))
    sens = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1); ppv = tp / max(tp + fp, 1); npv = tn / max(tn + fn, 1)
    return {"Model": name, "Split": split, "Threshold": thr,
            "AUROC": roc_auc_score(y, pp), "AUPRC": average_precision_score(y, pp),
            "Utility": norm_utility(pred, idx), "Sensitivity": sens, "Specificity": spec, "PPV": ppv, "NPV": npv,
            "F1": 2 * ppv * sens / max(ppv + sens, 1e-12), "BalancedAccuracy": (sens + spec) / 2,
            "AlarmRate": float(pred.mean()), "Brier": float(np.mean((pp - y) ** 2)), "ECE": ece_score(y, pp),
            "NLL": float(-np.mean(y * np.log(pp) + (1 - y) * np.log(1 - pp))),
            "CalSlope": cal_slope(y, pp), "CITL": float(pp.mean() - y.mean())}

def select_threshold(p, idx):
    """Utility-optimal threshold on the given (validation) rows."""
    q = np.unique(np.quantile(p[idx], np.linspace(0.50, 0.999, 120)))
    util = np.array([norm_utility(p[idx] >= t, idx) for t in q])
    return float(q[int(np.nanargmax(util))]), pd.DataFrame({"Threshold": q, "Utility": util})

class TemperatureCal:
    name = "temperature"
    def fit(self, p, y):
        z = logit(p); y = np.asarray(y)
        res = minimize_scalar(lambda T: -np.mean(y * np.log(np.clip(sigmoid(z / T), 1e-7, 1)) +
                                                  (1 - y) * np.log(np.clip(1 - sigmoid(z / T), 1e-7, 1))),
                              bounds=(0.05, 20), method="bounded")
        self.T = float(res.x); return self
    def transform(self, p): return sigmoid(logit(p) / self.T)
class PlattCal:
    name = "platt"
    def fit(self, p, y):
        self.lr = LogisticRegression(C=1e6, max_iter=500).fit(logit(p).reshape(-1, 1), y); return self
    def transform(self, p): return self.lr.predict_proba(logit(p).reshape(-1, 1))[:, 1]
class IsotonicCal:
    name = "isotonic"
    def fit(self, p, y):
        self.iso = IsotonicRegression(out_of_bounds="clip", y_min=1e-6, y_max=1 - 1e-6).fit(p, y); return self
    def transform(self, p): return self.iso.predict(p)
CALIBRATORS = {"temperature": TemperatureCal, "platt": PlattCal, "isotonic": IsotonicCal}
mark("06_utility_metrics")


In [ ]:
# Cell 7 — Patient-wise stratified splits, model factory and the checkpointed core pipeline
PAT_SITE = SITE[PAT_START]
PAT_ID = PID[PAT_START]
PAT_A = np.flatnonzero(PAT_SITE == "A"); PAT_B = np.flatnonzero(PAT_SITE == "B")

def stratified_partition(pcodes, fracs, seed):
    """Split patient codes into len(fracs) groups, stratified by septic status."""
    r = np.random.default_rng(seed); groups = [[] for _ in fracs]
    for stratum in (pcodes[SEPTIC_P[pcodes]], pcodes[~SEPTIC_P[pcodes]]):
        s = r.permutation(stratum); cuts = np.round(np.cumsum(fracs)[:-1] * len(s)).astype(int)
        for g, part in zip(groups, np.split(s, cuts)):
            g.extend(part.tolist())
    return [np.sort(np.array(g, dtype=int)) for g in groups]

def rows_of(pcodes):
    m = np.zeros(N_PAT, bool); m[pcodes] = True
    return np.flatnonzero(m[PCODE])

# Site B: fixed partition (main seed) into B_CAL (recalibration experiments only) and B_TEST
B_CAL_P, B_TE_P = stratified_partition(PAT_B, (B_CAL_FRAC, 1 - B_CAL_FRAC), SEED)
B_CAL, B_TE = rows_of(B_CAL_P), rows_of(B_TE_P)

def make_split(seed):
    tr, vf, vc, te = stratified_partition(PAT_A, SPLIT_FRACS, seed)
    sp = {"TR": rows_of(tr), "VF": rows_of(vf), "VC": rows_of(vc), "TE": rows_of(te), "B_CAL": B_CAL, "B_TE": B_TE,
          "pats": {"TR": tr, "VF": vf, "VC": vc, "TE": te, "B_CAL": B_CAL_P, "B_TE": B_TE_P}}
    P = sp["pats"]
    for a in P:
        for b in P:
            if a < b:
                assert np.intersect1d(P[a], P[b]).size == 0, f"patient overlap between {a} and {b}"
    return sp

def split_table(sp):
    rows = []
    for k in ["TR", "VF", "VC", "TE", "B_CAL", "B_TE"]:
        pc = sp["pats"][k]; ri = sp[k]
        rows.append({"Split": k, "Site": "A" if k in ("TR", "VF", "VC", "TE") else "B", "Patients": len(pc),
                     "SepticPatients": int(SEPTIC_P[pc].sum()), "Rows": len(ri), "PositiveRows": int(Y[ri].sum()),
                     "Role": {"TR": "model fitting", "VF": "ensemble weights, calibrator, thresholds, early stopping",
                              "VC": "conformal quantiles only", "TE": "untouched internal test",
                              "B_CAL": "site-B recalibration experiments only", "B_TE": "untouched external test"}[k]})
    return pd.DataFrame(rows)

def make_models(seed, ytr):
    pos = max(int(ytr.sum()), 1); neg = len(ytr) - pos
    imp = lambda: SimpleImputer(strategy="median")
    M = {"Logistic": Pipeline([("imp", imp()), ("sc", StandardScaler()),
                               ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=seed))]),
         "RandomForest": Pipeline([("imp", imp()), ("clf", RandomForestClassifier(
             n_estimators=200, min_samples_leaf=3, max_samples=0.3, class_weight="balanced", n_jobs=-1, random_state=seed))]),
         "ExtraTrees": Pipeline([("imp", imp()), ("clf", ExtraTreesClassifier(
             n_estimators=300, min_samples_leaf=3, bootstrap=True, max_samples=0.3, class_weight="balanced",
             n_jobs=-1, random_state=seed))])}
    if HAVE_LGB:   # native NaN handling: missingness kept informative
        M["LightGBM"] = LGBMClassifier(n_estimators=350, learning_rate=0.04, num_leaves=31, subsample=0.8,
                                       subsample_freq=1, colsample_bytree=0.8, class_weight="balanced",
                                       verbosity=-1, n_jobs=-1, random_state=seed)
    else:
        M["HistGB"] = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced", random_state=seed)
    if HAVE_XGB:
        M["XGBoost"] = XGBClassifier(n_estimators=350, max_depth=6, learning_rate=0.04, subsample=0.9,
                                     colsample_bytree=0.8, tree_method="hist", scale_pos_weight=neg / pos,
                                     eval_metric="logloss", n_jobs=-1, random_state=seed)
    return M

def run_core(seed, main=False):
    """Fit all members on TR, predict VF/VC/TE/B_CAL/B_TE; checkpointed so a disconnect resumes here."""
    sp = make_split(seed)
    eval_idx = np.concatenate([sp["VF"], sp["VC"], sp["TE"], sp["B_CAL"], sp["B_TE"]])
    # Checkpoint PER MEMBER: each fitted model (main seed) and its predictions are saved to Drive the
    # moment that member finishes, so a disconnect during the 4th model does not lose the first three.
    R = {"TE": sp["TE"], "P": {}, "eff": []}
    Xtr = ytr = None
    bench = sp["TE"][:min(20000, len(sp["TE"]))]
    for name in make_models(seed, Y[sp["TR"]]):
        mfile = ckpt_path(f"member_{name}_seed{seed}_{FEAT_HASH}")
        model_file = SUB["models"] / f"{name}_seed{seed}.joblib"
        if mfile.exists() and (not main or model_file.exists()):
            obj = joblib.load(mfile)
            if np.array_equal(obj["TE"], sp["TE"]):
                R["P"][name] = obj["p"]; R["eff"].append(obj["eff"])
                log(f"seed {seed}: {name} loaded from checkpoint"); continue
        if Xtr is None:
            Xtr, ytr = X[sp["TR"]], Y[sp["TR"]]
        m = make_models(seed, ytr)[name]
        t0 = time.perf_counter(); m.fit(Xtr, ytr); fit_s = time.perf_counter() - t0
        p = np.full(len(Y), np.nan, np.float32)
        for chunk in np.array_split(eval_idx, max(1, len(eval_idx) // 200000)):
            p[chunk] = m.predict_proba(X[chunk])[:, 1]
        t0 = time.perf_counter(); m.predict_proba(X[bench]); inf = (time.perf_counter() - t0) / len(bench) * 1000 * 1000
        size = np.nan
        if main:
            atomic_dump(m, model_file); size = model_file.stat().st_size / 1024 ** 2
        eff = {"Model": name, "Fit_s": fit_s, "Infer_ms_per_1000_rows": inf, "ModelSize_MB": size}
        atomic_dump({"TE": sp["TE"], "p": p, "eff": eff}, mfile)
        R["P"][name] = p; R["eff"].append(eff)
        log(f"seed {seed}: {name} fitted in {fit_s:.0f}s and checkpointed")
        del m; gc.collect()
    del Xtr; gc.collect()
    mark(f"07_members_seed{seed}", members=list(R["P"]))
    R["split"] = sp
    VF = sp["VF"]; yvf = Y[VF]
    names = list(R["P"])
    ap = {n: average_precision_score(yvf, R["P"][n][VF]) for n in names}
    w = {n: max(ap[n], 1e-6) for n in names}; s = sum(w.values()); w = {n: v / s for n, v in w.items()}
    R["weights"], R["val_ap"] = w, ap
    R["P"]["Ensemble_Equal"] = (sum(R["P"][n] for n in names) / len(names)).astype(np.float32)
    R["P"]["Ensemble_Raw"] = sum(w[n] * R["P"][n] for n in names).astype(np.float32)
    R["calibrators"] = {k: C().fit(R["P"]["Ensemble_Raw"][VF], yvf) for k, C in CALIBRATORS.items()}
    sg = np.full(len(Y), np.nan, np.float32)
    sg[eval_idx] = R["calibrators"][PRIMARY_CALIBRATOR].transform(R["P"]["Ensemble_Raw"][eval_idx])
    R["P"]["SepsisGuard_CP"] = sg
    R["best_single"] = max(names, key=lambda n: ap[n])
    R["members"] = names
    R["thr"] = {}; R["thr_curves"] = {}
    for n, p in R["P"].items():
        R["thr"][n], R["thr_curves"][n] = select_threshold(p, VF)
    return R


In [ ]:
# Cell 8 — Main seed: fit members, build SepsisGuard-CP, evaluate on untouched A-test and B-test
R0 = run_core(SEED, main=True)
SP = R0["split"]; P = R0["P"]; THR = R0["thr"]
TR, VF, VC, TE = SP["TR"], SP["VF"], SP["VC"], SP["TE"]
MEMBERS = R0["members"]; W = R0["weights"]; CAL = R0["calibrators"][PRIMARY_CALIBRATOR]
MODELS = {n: joblib.load(SUB["models"] / f"{n}_seed{SEED}.joblib") for n in MEMBERS}
EVAL_IDX = np.concatenate([VF, VC, TE, B_CAL, B_TE])
# Fingerprint of the fitted ensemble: downstream checkpoints are reused only if the models are identical.
MODEL_SIG = hashlib.md5(np.round(P["SepsisGuard_CP"][EVAL_IDX].astype(np.float64), 6).tobytes()).hexdigest()[:10]
log(f"model signature {MODEL_SIG}")

def predict_sg(Xn, return_members=False):
    """SepsisGuard-CP probability for an arbitrary feature matrix (used by robustness and permutation tests)."""
    mp = {n: MODELS[n].predict_proba(Xn)[:, 1] for n in MEMBERS}
    p = CAL.transform(sum(W[n] * mp[n] for n in MEMBERS))
    return (p, mp) if return_members else p

# Leakage assertions on the final roles
assert np.intersect1d(SP["pats"]["TR"], SP["pats"]["TE"]).size == 0
assert np.isnan(P["SepsisGuard_CP"][TR]).all(), "training rows must never receive evaluation predictions"
split_df = split_table(SP)
save_table(split_df, "T03_Patient_Splits"); display(split_df)

ORDER = MEMBERS + ["Ensemble_Equal", "Ensemble_Raw", "SepsisGuard_CP"]
perf = pd.DataFrame([evaluate(n, s, P[n], idx, THR[n]) for s, idx in [("A_test", TE), ("B_test", B_TE)] for n in ORDER])
save_table(perf, "T05_Performance_A_test_B_test")
weights_df = pd.DataFrame({"Member": MEMBERS, "VA_FIT_AUPRC": [R0["val_ap"][n] for n in MEMBERS],
                           "EnsembleWeight": [W[n] for n in MEMBERS], "UtilityThreshold": [THR[n] for n in MEMBERS]})
save_table(weights_df, "T06a_Ensemble_Weights_Thresholds")
curves = pd.concat([c.assign(Model=n) for n, c in R0["thr_curves"].items()], ignore_index=True)
save_table(curves, "T06b_VAFIT_Utility_vs_Threshold")
display(perf.round(4))
mark("08_main_evaluation")


In [ ]:
# Cell 8b — Clinical score baselines: qSOFA, SIRS, NEWS2-style and partial SOFA, computed hourly
# from the same past-only last-observed values. Components that cannot be derived from this dataset
# (mental status / GCS, PaO2-based respiratory SOFA, vasopressors, urine output) are omitted, so every
# score is a PARTIAL version; an unmeasured component contributes 0 points, as in bedside practice.
# Scores are ranked risk measures, not probabilities: their calibration columns are not interpreted.
CLIN_SCORES = []
try:
    _col = {n: i for i, n in enumerate(FEATURES)}
    def _last(c): return X[:, _col[f"{c}_last"]].astype(np.float64)
    HR_, RR_, SBP_, MAP_, T_, O2_, FIO2_, WBC_, PLT_, BILI_, CREA_, PACO2_ = (_last(c) for c in
        ["HR", "Resp", "SBP", "MAP", "Temp", "O2Sat", "FiO2", "WBC", "Platelets", "Bilirubin_total", "Creatinine", "PaCO2"])
    def band(v, edges_points):
        """edges_points: list of (condition_function, points); first matching condition wins; NaN -> 0."""
        out = np.zeros(len(v))
        done = np.isnan(v)
        for cond, pts in edges_points:
            m = ~done & cond(v); out[m] = pts; done |= m
        return out
    with np.errstate(invalid="ignore"):
        qsofa = (RR_ >= 22).astype(float) + (SBP_ <= 100).astype(float)
        sirs = (((T_ > 38) | (T_ < 36)).astype(float) + (HR_ > 90).astype(float)
                + ((RR_ > 20) | (PACO2_ < 32)).astype(float) + ((WBC_ > 12) | (WBC_ < 4)).astype(float))
        supp_o2 = (((FIO2_ > 0.21) & (FIO2_ <= 1.0)) | (FIO2_ > 21)).astype(float) * 2
        news = (band(RR_, [(lambda v: v <= 8, 3), (lambda v: v <= 11, 1), (lambda v: v <= 20, 0), (lambda v: v <= 24, 2), (lambda v: v >= 25, 3)])
                + band(O2_, [(lambda v: v <= 91, 3), (lambda v: v <= 93, 2), (lambda v: v <= 95, 1), (lambda v: v >= 96, 0)])
                + supp_o2
                + band(T_, [(lambda v: v <= 35.0, 3), (lambda v: v <= 36.0, 1), (lambda v: v <= 38.0, 0), (lambda v: v <= 39.0, 1), (lambda v: v > 39.0, 2)])
                + band(SBP_, [(lambda v: v <= 90, 3), (lambda v: v <= 100, 2), (lambda v: v <= 110, 1), (lambda v: v <= 219, 0), (lambda v: v >= 220, 3)])
                + band(HR_, [(lambda v: v <= 40, 3), (lambda v: v <= 50, 1), (lambda v: v <= 90, 0), (lambda v: v <= 110, 1), (lambda v: v <= 130, 2), (lambda v: v > 130, 3)]))
        sofa = (band(PLT_, [(lambda v: v < 20, 4), (lambda v: v < 50, 3), (lambda v: v < 100, 2), (lambda v: v < 150, 1)])
                + band(BILI_, [(lambda v: v >= 12, 4), (lambda v: v >= 6, 3), (lambda v: v >= 2, 2), (lambda v: v >= 1.2, 1)])
                + band(CREA_, [(lambda v: v >= 5, 4), (lambda v: v >= 3.5, 3), (lambda v: v >= 2, 2), (lambda v: v >= 1.2, 1)])
                + (MAP_ < 70).astype(float))
    SCORE_MAX = {"qSOFA_partial": 2, "SIRS": 4, "NEWS2_partial": 18, "SOFA_partial": 13}
    for name, v in [("qSOFA_partial", qsofa), ("SIRS", sirs), ("NEWS2_partial", news), ("SOFA_partial", sofa)]:
        P[name] = (v / SCORE_MAX[name]).astype(np.float32)        # rank-preserving rescale to [0, 1]
        THR[name] = select_threshold(P[name], VF)[0]
        CLIN_SCORES.append(name)
    del HR_, RR_, SBP_, MAP_, T_, O2_, FIO2_, WBC_, PLT_, BILI_, CREA_, PACO2_; gc.collect()
    clin_df = pd.DataFrame([evaluate(n, s, P[n], idx, THR[n]) for s, idx in [("A_test", TE), ("B_test", B_TE)]
                            for n in CLIN_SCORES + ["SepsisGuard_CP"]])
    clin_df["Note"] = np.where(clin_df.Model.isin(CLIN_SCORES), "partial clinical score; calibration columns not interpretable", "")
    save_table(clin_df, "T05b_Clinical_Score_Baselines"); display(clin_df[["Model", "Split", "AUROC", "AUPRC", "Utility", "Sensitivity", "Specificity"]].round(4))
    mark("08b_clinical_scores")
except Exception as _e:
    stage_failed("08b_clinical_scores", _e)
    CLIN_SCORES = [c for c in CLIN_SCORES if c in P]


In [ ]:
# Cell 8c — Bounded hyperparameter search for the strongest baseline: LightGBM tuned on VA_FIT only
# Random search over a small pre-specified space with early stopping on VA_FIT AUPRC; the best configuration
# is evaluated once on the untouched A and B test sets. The ensemble members keep their fixed defaults, so
# this answers "would a tuned single model beat SepsisGuard-CP?" Each trial is checkpointed.
TUNED = None
try:
    if HAVE_LGB:
        rs_h = np.random.default_rng(SEED)
        SPACE = dict(num_leaves=[15, 31, 63, 127], learning_rate=[0.03, 0.06, 0.1], min_child_samples=[20, 100, 400],
                     colsample_bytree=[0.5, 0.8], reg_lambda=[0.0, 1.0, 10.0])
        configs = [{k: v[int(rs_h.integers(len(v)))] for k, v in SPACE.items()} for _ in range(CFG["hpo_trials"])]
        Xtr_h, ytr_h, Xvf_h = None, Y[TR], None
        hpo_rows = []; best = None
        for i, cfg in enumerate(configs):
            def _trial():
                global Xtr_h, Xvf_h
                if Xtr_h is None:
                    Xtr_h, Xvf_h = X[TR], X[VF]
                m = LGBMClassifier(n_estimators=1500, subsample=0.8, subsample_freq=1, class_weight="balanced",
                                   verbosity=-1, n_jobs=-1, random_state=SEED, **cfg)
                t0 = time.perf_counter()
                m.fit(Xtr_h, ytr_h, eval_set=[(Xvf_h, Y[VF])], eval_metric="average_precision",
                      callbacks=[lgb.early_stopping(100, verbose=False)])
                return {"model": m, "vf_ap": average_precision_score(Y[VF], m.predict_proba(Xvf_h)[:, 1]),
                        "best_iter": int(m.best_iteration_ or m.n_estimators), "fit_s": time.perf_counter() - t0}
            res = cached(f"hpo_lgbm_trial{i}", _trial, key=(FEAT_HASH, SEED, tuple(sorted(cfg.items()))))
            hpo_rows.append({"Trial": i, **cfg, "BestIteration": res["best_iter"], "VA_FIT_AUPRC": res["vf_ap"], "Fit_s": res["fit_s"]})
            if best is None or res["vf_ap"] > best["vf_ap"]:
                best = res
            log(f"HPO trial {i}: VA_FIT AUPRC={res['vf_ap']:.4f}")
        Xtr_h = Xvf_h = None; gc.collect()
        TUNED = best["model"]
        pt_ = np.full(len(Y), np.nan, np.float32)
        for idx in (VF, VC, TE, B_CAL, B_TE):
            pt_[idx] = TUNED.predict_proba(X[idx])[:, 1]
        P["LightGBM_tuned"] = pt_; THR["LightGBM_tuned"] = select_threshold(pt_, VF)[0]
        hpo_df = pd.DataFrame(hpo_rows).sort_values("VA_FIT_AUPRC", ascending=False)
        hpo_df["Selected"] = hpo_df.VA_FIT_AUPRC == hpo_df.VA_FIT_AUPRC.max()
        save_table(hpo_df, "T05c_LightGBM_HPO_Trials")
        tuned_df = pd.DataFrame([evaluate(n, s, P[n], idx, THR[n]) for s, idx in [("A_test", TE), ("B_test", B_TE)]
                                 for n in ["LightGBM", "LightGBM_tuned", "SepsisGuard_CP"]])
        save_table(tuned_df, "T05d_Tuned_LightGBM_vs_SepsisGuard"); display(tuned_df[["Model", "Split", "AUROC", "AUPRC", "Utility"]].round(4))
    else:
        print("[WARN] LightGBM unavailable; tuned-baseline comparison skipped.")
    mark("08c_tuned_baseline")
except Exception as _e:
    stage_failed("08c_tuned_baseline", _e)


In [ ]:
# Cell 9 — Calibration: raw ensemble vs temperature, Platt (primary, pre-declared) and isotonic; all fitted on VA_FIT only
cal_rows = []
for split, idx in [("A_test", TE), ("B_test", B_TE)]:
    y = Y[idx].astype(int)
    variants = {"Ensemble_Raw": P["Ensemble_Raw"][idx]}
    for k, c in R0["calibrators"].items():
        variants[f"Ensemble+{k}"] = c.transform(P["Ensemble_Raw"][idx])
    for name, p in variants.items():
        p = np.clip(p, 1e-7, 1 - 1e-7)
        cal_rows.append({"Split": split, "Variant": name, "Primary": name == f"Ensemble+{PRIMARY_CALIBRATOR}",
                         "ECE": ece_score(y, p), "Brier": float(np.mean((p - y) ** 2)),
                         "NLL": float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))),
                         "CalSlope": cal_slope(y, p), "CITL": float(p.mean() - y.mean()), "MeanPred": float(p.mean()),
                         "ObservedRate": float(y.mean())})
cal_df = pd.DataFrame(cal_rows)
save_table(cal_df, "T07_Calibration"); display(cal_df.round(4))

def reliability_table(y, p, bins=10):
    q = np.unique(np.quantile(p, np.linspace(0, 1, bins + 1)))
    b = np.clip(np.searchsorted(q, p, side="right") - 1, 0, len(q) - 2)
    return pd.DataFrame({"bin": np.arange(len(q) - 1),
                         "mean_pred": np.bincount(b, p, len(q) - 1) / np.maximum(np.bincount(b, minlength=len(q) - 1), 1),
                         "obs_rate": np.bincount(b, y, len(q) - 1) / np.maximum(np.bincount(b, minlength=len(q) - 1), 1),
                         "n": np.bincount(b, minlength=len(q) - 1)})
mark("09_calibration")


In [ ]:
# Cell 10 — Split conformal prediction sets: row-marginal, Mondrian (class-conditional) and patient-level
# Quantiles are computed ONLY on VA_CONF, which no fitting step has touched.
# Row-marginal / Mondrian: guarantee for a randomly drawn patient-hour (rows are clustered within patients,
#   so this is row-level coverage).
# Patient-level: nonconformity of a patient = max over that patient's hours. Patients are exchangeable,
#   so with probability >= 1 - alpha EVERY hour of a new patient's stay is covered.
# With ~2% septic hours, marginal and patient-level sets can reach 90% coverage without covering septic
#   hours, so the PRIMARY method is Mondrian (class-conditional). 'patient_mondrian' is the class-conditional patient-level version: for
#   each class c, a patient's score is the max over that patient's class-c hours, and the quantile is taken
#   over patients, so all class-c hours of a new patient are covered with probability >= 1 - alpha.
def conf_quantile(scores, alpha=ALPHA):
    s = np.sort(np.asarray(scores)); n = len(s); k = int(np.ceil((n + 1) * (1 - alpha)))
    return 1.0 if (n == 0 or k > n) else float(s[k - 1])

def nonconf(p, y):            # 1 - probability assigned to the true class
    return np.where(y == 1, 1 - p, p)

def fit_conformal(p, idx, alpha=ALPHA):
    y = Y[idx]; s = nonconf(p[idx], y)
    q_marg = conf_quantile(s, alpha)
    q_mond = {c: conf_quantile(s[y == c], alpha) for c in (0, 1)}
    starts = np.r_[0, np.flatnonzero(np.diff(PCODE[idx])) + 1]
    pat_max = np.maximum.reduceat(s, starts)
    q_pat = conf_quantile(pat_max, alpha)
    q_pm = {}
    for c in (0, 1):
        sc = np.where(y == c, s, -np.inf)
        pm = np.maximum.reduceat(sc, starts)
        q_pm[c] = conf_quantile(pm[np.isfinite(pm)], alpha)
    return {"mondrian": (q_mond[0], q_mond[1]), "patient_mondrian": (q_pm[0], q_pm[1]),
            "marginal": (q_marg, q_marg), "patient": (q_pat, q_pat)}
PRIMARY_CP = "mondrian"

def conformal_sets(p, q0, q1):
    return np.c_[p <= q0, (1 - p) <= q1]          # include class 0 if score_0 = p <= q0; class 1 if 1-p <= q1

def conformal_report(p, idx, Q, split):
    y = Y[idx].astype(int); pp = p[idx]; rows = []
    starts = np.r_[0, np.flatnonzero(np.diff(PCODE[idx])) + 1]
    for method, (q0, q1) in Q.items():
        S = conformal_sets(pp, q0, q1); size = S.sum(1); cov = S[np.arange(len(y)), y]
        pat_cov = np.minimum.reduceat(cov.astype(int), starts)
        # class-conditional patient-level coverage: all of a patient's class-c hours covered
        cov1 = np.where(y == 1, cov, True); cov0 = np.where(y == 0, cov, True)
        has1 = np.add.reduceat((y == 1).astype(int), starts) > 0; has0 = np.add.reduceat((y == 0).astype(int), starts) > 0
        pc1 = np.minimum.reduceat(cov1.astype(int), starts); pc0 = np.minimum.reduceat(cov0.astype(int), starts)
        rows.append({"Split": split, "Method": method, "Primary": method == PRIMARY_CP, "Nominal": 1 - ALPHA, "q0": q0, "q1": q1,
                     "RowCoverage": cov.mean(), "Coverage_NonSepsisRows": cov[y == 0].mean(),
                     "Coverage_SepsisRows": cov[y == 1].mean() if (y == 1).any() else np.nan,
                     "PatientSimultaneousCoverage": pat_cov.mean(),
                     "PatientCov_SepticHours": pc1[has1].mean() if has1.any() else np.nan,
                     "PatientCov_NonSepticHours": pc0[has0].mean() if has0.any() else np.nan,
                     "MeanSetSize": size.mean(),
                     "SingletonRate": np.mean(size == 1), "AmbiguousRate": np.mean(size == 2), "EmptyRate": np.mean(size == 0)})
    return rows

def coverage_by_onset(p, idx, Q, split):
    y = Y[idx].astype(int); dt = POS[idx] - TSEP[idx]; sep = SEPTIC[idx]
    bins = [(-np.inf, -12), (-12, -6), (-6, 0), (0, 3), (3, np.inf)]
    labels = ["<= -12 h", "(-12,-6] h", "(-6,0] h", "(0,3] h", "> 3 h"]
    rows = []
    for method, (q0, q1) in Q.items():
        cov = conformal_sets(p[idx], q0, q1)[np.arange(len(y)), y]
        for (lo, hi), lab in zip(bins, labels):
            m = sep & (dt > lo) & (dt <= hi)
            rows.append({"Split": split, "Method": method, "HoursFromOnset": lab, "Rows": int(m.sum()),
                         "Coverage": cov[m].mean() if m.any() else np.nan})
        m = ~sep
        rows.append({"Split": split, "Method": method, "HoursFromOnset": "non-septic patients", "Rows": int(m.sum()),
                     "Coverage": cov[m].mean() if m.any() else np.nan})
    return rows

SG = P["SepsisGuard_CP"]
Q_A = fit_conformal(SG, VC)
conf_rows = conformal_report(SG, TE, Q_A, "A_test") + conformal_report(SG, B_TE, Q_A, "B_test_unchanged")
conf_df = pd.DataFrame(conf_rows)
onset_df = pd.DataFrame(coverage_by_onset(SG, TE, Q_A, "A_test") + coverage_by_onset(SG, B_TE, Q_A, "B_test_unchanged"))
save_table(conf_df, "T08_Conformal_Summary"); save_table(onset_df, "T10_Conformal_By_Hours_From_Onset")
display(conf_df.round(4))
mark("10_conformal")


In [ ]:
# Cell 11 — Patient-level alarm timeliness and alarm burden
# Reference time is t_sepsis = first positive Challenge label + 6 h (the Challenge's own definition), which
# is a Sepsis-3 operationalisation and not bedside onset. Patients labelled positive from their first ICU
# hour (t_sepsis <= 6 h) cannot be warned 6 h ahead inside the ICU record and are reported separately.
def alarm_events(p, idx, thr):
    alarm = p[idx] >= thr; pc = PCODE[idx]; pos = POS[idx]
    starts = np.r_[0, np.flatnonzero(np.diff(pc)) + 1]
    pats = pc[starts]; nrows = np.diff(np.r_[starts, len(idx)])
    prev = np.r_[False, alarm[:-1]]; prev[starts] = False
    episodes = np.add.reduceat((alarm & ~prev).astype(int), starts)
    alarm_hours = np.add.reduceat(alarm.astype(int), starts)
    ts = TSEP_P[pats]
    first_alarm = np.minimum.reduceat(np.where(alarm, pos, BIG), starts).astype(float)
    first_alarm[first_alarm >= BIG] = np.nan
    dt = pos - TSEP[idx]
    timely = np.add.reduceat((alarm & (dt >= DT_EARLY) & (dt <= DT_LATE)).astype(int), starts) > 0
    early_ok = np.minimum.reduceat(np.where(alarm & (dt <= DT_LATE), pos, BIG), starts).astype(float)
    early_ok[early_ok >= BIG] = np.nan
    # Clinically bounded lead time = first alarm within the 48 h before t_sepsis (up to t_sepsis + 3 h)
    in48 = alarm & (dt >= -LEAD_WINDOW_H) & (dt <= DT_LATE)
    first48 = np.minimum.reduceat(np.where(in48, pos, BIG), starts).astype(float)
    first48[first48 >= BIG] = np.nan
    return pd.DataFrame({"patient": PAT_ID[pats], "septic": SEPTIC_P[pats], "rows": nrows,
                         "onset_at_admission": SEPTIC_P[pats] & (FIRST_LAB_P[pats] == 0),
                         "t_sepsis": ts, "first_alarm": first_alarm, "detected_by_t_sepsis_plus3": ~np.isnan(early_ok),
                         "timely_window_alarm": timely, "lead_h": ts - early_ok,
                         "detected_48h_window": ~np.isnan(first48), "lead48_h": ts - first48,
                         "premature_first_alarm": first_alarm < ts + DT_EARLY,
                         "alarm_hours": alarm_hours, "alarm_episodes": episodes, "any_alarm": alarm_hours > 0})

def summarise_events(ev, model, split):
    sep = ev[ev.septic & ~ev.onset_at_admission]; non = ev[~ev.septic]
    days_non = non.rows.sum() / 24
    lead = sep.lead_h.dropna(); lead48 = sep.lead48_h.dropna()
    return {"Model": model, "Split": split, "SepticPatients_evaluable": len(sep),
            "SepticPatients_onset_at_admission": int(ev.onset_at_admission.sum()), "NonSepticPatients": len(non),
            "DetectedBy_tsepsis+3h": sep.detected_by_t_sepsis_plus3.mean(), "TimelyWindowDetection_[-12,+3]h": sep.timely_window_alarm.mean(),
            "MedianLead_h": lead.median(), "LeadIQR_low": lead.quantile(.25), "LeadIQR_high": lead.quantile(.75),
            "PrematureFirstAlarm_frac": sep.premature_first_alarm.mean(),
            "Detected_in_48h_window": sep.detected_48h_window.mean(), "MedianLead48_h": lead48.median(),
            "Lead48_IQR_low": lead48.quantile(.25), "Lead48_IQR_high": lead48.quantile(.75),
            "NonSeptic_anyAlarm_frac": non.any_alarm.mean(),
            "NonSeptic_alarmHours_per_patientDay": non.alarm_hours.sum() / max(days_non, 1e-9),
            "NonSeptic_alarmEpisodes_per_patientDay": non.alarm_episodes.sum() / max(days_non, 1e-9),
            "AlarmEpisodes_per_100_patientHours_all": 100 * ev.alarm_episodes.sum() / ev.rows.sum()}

LEAD_WINDOW_H = 48
BEST = R0["best_single"]
# Operating points: the utility-optimal threshold plus fixed row-level specificities, all chosen on VA_FIT only
def spec_threshold(p, idx, spec):
    return float(np.quantile(p[idx][Y[idx] == 0], spec))
OP_THR = {}
for n in ["SepsisGuard_CP", BEST, "Ensemble_Equal"]:
    OP_THR[(n, "utility_optimal")] = THR[n]
    for sp_ in (0.90, 0.95, 0.98):
        OP_THR[(n, f"specificity_{int(sp_*100)}")] = spec_threshold(P[n], VF, sp_)
ev_rows = []; EVENTS = {}
for split, idx in [("A_test", TE), ("B_test", B_TE)]:
    for (n, op), thr in OP_THR.items():
        ev = alarm_events(P[n], idx, thr); EVENTS[(n, split, op)] = ev
        r = summarise_events(ev, n, split); r["OperatingPoint"] = op; r["Threshold"] = thr
        r["RowSensitivity"] = float(np.mean(P[n][idx][Y[idx] == 1] >= thr)); r["RowSpecificity"] = float(np.mean(P[n][idx][Y[idx] == 0] < thr))
        ev_rows.append(r)
events_df = pd.DataFrame(ev_rows)
EVENTS[("SepsisGuard_CP", "A_test")] = EVENTS[("SepsisGuard_CP", "A_test", "utility_optimal")]
EVENTS[("SepsisGuard_CP", "A_test")].to_csv(SUB["tables"] / "T11b_Patient_Events_SepsisGuard_A_test.csv", index=False)
save_table(events_df, "T11_Alarm_Timeliness_Burden"); display(events_df.round(3))
mark("11_alarm_events")


In [ ]:
# Cell 12 — Cross-hospital transfer A -> B: unchanged model vs local recalibration on B_CAL
# The members and ensemble weights are never refitted on B. Only the post-hoc layer is re-estimated on
# B_CAL patients, which are disjoint from B_TEST.
raw_bcal = P["Ensemble_Raw"][B_CAL]; ybcal = Y[B_CAL]
xs_rows = []; XS_PROBS = {}
conditions = {"Unchanged (A calibration, A threshold)": (CAL, THR["SepsisGuard_CP"], False)}
for k in ["temperature", "platt", "isotonic"]:
    conditions[f"B_CAL {k} recalibration, A threshold"] = (CALIBRATORS[k]().fit(raw_bcal, ybcal), None, False)
conditions["B_CAL platt recalibration + B_CAL threshold"] = (CALIBRATORS["platt"]().fit(raw_bcal, ybcal), None, True)
for cname, (cal, thr, reselect) in conditions.items():
    p = np.full(len(Y), np.nan); p[B_CAL] = cal.transform(raw_bcal); p[B_TE] = cal.transform(P["Ensemble_Raw"][B_TE])
    if thr is None:
        thr = select_threshold(p, B_CAL)[0] if reselect else THR["SepsisGuard_CP"]
    XS_PROBS[cname] = p
    r = evaluate("SepsisGuard_CP", "B_test", p, B_TE, thr); r["Condition"] = cname
    xs_rows.append(r)
r = evaluate("SepsisGuard_CP", "A_test", SG, TE, THR["SepsisGuard_CP"]); r["Condition"] = "Internal reference (A test)"
xs_rows.insert(0, r)
xs_df = pd.DataFrame(xs_rows)[["Condition", "Split", "Threshold", "AUROC", "AUPRC", "Utility", "Sensitivity",
                               "Specificity", "PPV", "ECE", "Brier", "CalSlope", "CITL", "AlarmRate"]]
save_table(xs_df, "T12_CrossSite_Recalibration"); display(xs_df.round(4))

# Conformal under transfer: A quantiles vs quantiles re-estimated on B_CAL (recalibrated probabilities)
p_recal = XS_PROBS["B_CAL platt recalibration, A threshold"]
Q_B = fit_conformal(p_recal, B_CAL)
xs_conf = pd.DataFrame(conformal_report(SG, B_TE, Q_A, "B_test: A model + A quantiles")
                       + conformal_report(p_recal, B_TE, Q_B, "B_test: B_CAL Platt + B_CAL quantiles"))
save_table(xs_conf, "T13_CrossSite_Conformal"); display(xs_conf.round(4))
mark("12_cross_site")


In [ ]:
# Cell 13 — Label-budget curve: how many local (site-B) patients restore calibration, coverage and utility?
# For each budget n, n patients are drawn from B_CAL (repeated with different draws); Platt recalibration,
# the utility threshold and the conformal quantile are re-estimated on those n patients only, and everything
# is evaluated on the fixed, untouched B_TEST.
# Isolated stage: on failure the traceback is saved to logs/ and the run continues.
try:
    ybte = Y[B_TE].astype(int); raw_bte = P["Ensemble_Raw"][B_TE]
    def budget_eval(sub_pats, rep_seed):
        rows = rows_of(sub_pats); yy = Y[rows]
        if len(np.unique(yy)) < 2:
            return None
        cal = PlattCal().fit(P["Ensemble_Raw"][rows], yy)
        p = np.full(len(Y), np.nan); p[rows] = cal.transform(P["Ensemble_Raw"][rows]); p[B_TE] = cal.transform(raw_bte)
        thr = select_threshold(p, rows)[0]
        q = conf_quantile(nonconf(p[rows], yy))
        S = conformal_sets(p[B_TE], q, q); cov = S[np.arange(len(ybte)), ybte]
        Qb = fit_conformal(p, rows)                        # class-conditional (Mondrian) coverage
        # Re-select the 95%-specificity alarm threshold on the n local patients only
        thr95 = float(np.quantile(p[rows][yy == 0], 0.95))
        ev95 = alarm_events(p, B_TE, thr95); s95 = ev95[ev95.septic & ~ev95.onset_at_admission]; n95 = ev95[~ev95.septic]
        Sm = conformal_sets(p[B_TE], *Qb["mondrian"]); covm = Sm[np.arange(len(ybte)), ybte]
        return {"Spec95_RealizedSpecificity": float(np.mean(p[B_TE][ybte == 0] < thr95)),
                "Spec95_Detected48h": float(s95.detected_48h_window.mean()),
                "Spec95_FalseEpisodesPerDay": float(n95.alarm_episodes.sum() / max(n95.rows.sum() / 24, 1e-9)),
                "Mondrian_SepsisRowCoverage": covm[ybte == 1].mean(), "Mondrian_NonSepsisRowCoverage": covm[ybte == 0].mean(),
                "Mondrian_MeanSetSize": Sm.sum(1).mean(), "LocalSepticPatients": int(SEPTIC_P[sub_pats].sum()),"ECE": ece_score(ybte, p[B_TE]), "CalSlope": cal_slope(ybte, p[B_TE]), "CITL": float(p[B_TE].mean() - ybte.mean()),
                "Brier": float(np.mean((p[B_TE] - ybte) ** 2)), "Utility": norm_utility(p[B_TE] >= thr, B_TE),
                "RowCoverage": cov.mean(), "SepsisRowCoverage": cov[ybte == 1].mean(), "MeanSetSize": S.sum(1).mean()}

    sizes = [b for b in BUDGETS if b < len(B_CAL_P)] + [len(B_CAL_P)]
    def _run_budget():
      budget_rows = []
      for n in sizes:
          reps = 1 if n == len(B_CAL_P) else CFG["budget_reps"]
          for rep in range(reps):
              r = np.random.default_rng(10_000 + 97 * n + rep)
              res = None; tries = 0
              while res is None and tries < 20:
                  res = budget_eval(np.sort(r.choice(B_CAL_P, size=n, replace=False)), rep); tries += 1
              if res is None:
                  continue
              budget_rows.append({"LocalPatients": n, "Rep": rep, **res})
      return budget_rows
    budget_rows = cached("label_budget_v82", _run_budget, key=(FEAT_HASH, MODEL_SIG, CFG["budget_reps"], tuple(sizes), len(B_CAL_P)))
    budget_raw = pd.DataFrame(budget_rows)
    ref_unchanged = {"ECE": ece_score(ybte, SG[B_TE]), "RowCoverage": float(xs_conf[(xs_conf.Method == "marginal") & (xs_conf.Split.str.contains("A model"))].RowCoverage.iloc[0])}
    budget_df = budget_raw.groupby("LocalPatients").agg(["mean", "std"])
    budget_df.columns = [f"{a}_{b}" for a, b in budget_df.columns]
    budget_df = budget_df.drop(columns=[c for c in budget_df.columns if c.startswith("Rep_")]).reset_index()
    budget_df["Unchanged_ECE"] = ref_unchanged["ECE"]; budget_df["Unchanged_RowCoverage"] = ref_unchanged["RowCoverage"]
    # Reference: the unchanged A model (A calibration + A Mondrian quantiles) evaluated on B_TEST
    Sm0 = conformal_sets(SG[B_TE], *Q_A["mondrian"]); cm0 = Sm0[np.arange(len(ybte)), ybte]
    budget_df["Unchanged_Mondrian_SepsisRowCoverage"] = cm0[ybte == 1].mean()
    _u95 = events_df[(events_df.Model == "SepsisGuard_CP") & (events_df.Split == "B_test") & (events_df.OperatingPoint == "specificity_95")].iloc[0]
    budget_df["Unchanged_Spec95_RealizedSpecificity"] = _u95.RowSpecificity
    budget_df["Unchanged_Spec95_Detected48h"] = _u95.Detected_in_48h_window
    budget_df["Recalibration_beats_unchanged_ECE"] = budget_df.ECE_mean <= budget_df.Unchanged_ECE
    _be = budget_df[budget_df.Recalibration_beats_unchanged_ECE]
    N_BREAKEVEN = int(_be.LocalPatients.min()) if len(_be) else None
    log(f"label budget: recalibration matches/beats the unchanged model's ECE from n = {N_BREAKEVEN} local patients")
    save_table(budget_raw, "T14b_Label_Budget_Raw_Reps"); save_table(budget_df, "T14_Label_Budget_Curve")
    display(budget_df.round(4))
    mark("13_label_budget")
except Exception as _e:
    stage_failed("13_label_budget", _e)
    budget_df = pd.DataFrame(); budget_raw = pd.DataFrame(); N_BREAKEVEN = None


In [ ]:
# Cell 14 — Ablations
# (a) Incremental feature-group ablation with the gradient-boosting member: each step adds ONE block.
# (b) Ensemble-component ablation: best single -> equal weights -> AUPRC weights -> + recalibration.
# Isolated stage: on failure the traceback is saved to logs/ and the run continues.
try:
    STEPS = [("Current + static context", ["static_context", "current"]),
             ("+ missingness indicators", ["missingness"]),
             ("+ last observed value", ["last_observed"]),
             ("+ 6 h window mean/SD", ["window_6h"]),
             ("+ 12/24 h window mean/SD", ["window_12_24h"]),
             ("+ deltas", ["delta"]),
             ("+ clinical composites (= full set)", ["clinical_composite"])]
    abl_model_name = "LightGBM" if HAVE_LGB else "HistGB"
    abl_rows = []; cols = []
    for step, fams in STEPS:
        for f in fams:
            cols += FAM_COLS[f]
        ckpt = SUB["checkpoints"] / f"ablation_{RUN_MODE}_{FEAT_HASH}_{len(cols)}.joblib"
        if ckpt.exists():
            pa = joblib.load(ckpt)
        else:
            m = make_models(SEED, Y[TR])[abl_model_name]
            m.fit(X[np.ix_(TR, cols)], Y[TR])
            pa = np.full(len(Y), np.nan, np.float32)
            for idx in (VF, TE, B_TE):
                pa[idx] = m.predict_proba(X[np.ix_(idx, cols)])[:, 1]
            atomic_dump(pa, ckpt)
        thr = select_threshold(pa, VF)[0]
        for split, idx in [("A_test", TE), ("B_test", B_TE)]:
            r = evaluate(abl_model_name, split, pa, idx, thr)
            abl_rows.append({"Step": step, "NumFeatures": len(cols), "Split": split, "AUROC": r["AUROC"], "AUPRC": r["AUPRC"],
                             "Utility": r["Utility"], "Sensitivity": r["Sensitivity"], "Specificity": r["Specificity"]})
        log(f"ablation step done: {step} ({len(cols)} features)")
    abl_df = pd.DataFrame(abl_rows)
    save_table(abl_df, "T15_Feature_Group_Ablation"); display(abl_df.round(4))

    ens_steps = [(f"Best single member ({BEST})", BEST), ("Equal-weight ensemble", "Ensemble_Equal"),
                 ("AUPRC-weighted ensemble (raw)", "Ensemble_Raw"), (f"+ {PRIMARY_CALIBRATOR} recalibration (SepsisGuard-CP)", "SepsisGuard_CP")]
    ens_rows = []
    for step, n in ens_steps:
        for split, idx in [("A_test", TE), ("B_test", B_TE)]:
            r = evaluate(n, split, P[n], idx, THR[n])
            ens_rows.append({"Step": step, "Split": split, **{k: r[k] for k in ["AUROC", "AUPRC", "Utility", "ECE", "Brier", "CITL"]}})
    ens_abl_df = pd.DataFrame(ens_rows)
    save_table(ens_abl_df, "T16_Ensemble_Component_Ablation"); display(ens_abl_df.round(4))
    mark("14_ablations")
except Exception as _e:
    stage_failed("14_ablations", _e)
    abl_df = pd.DataFrame(); ens_abl_df = pd.DataFrame()


In [ ]:
# Cell 15 — Robustness: perturb the RAW hourly measurements of test patients, then rebuild all features
# (v7 masked only the *_current/*_last columns, which left the rolling-window features unperturbed.)
# Isolated stage: on failure the traceback is saved to logs/ and the run continues.
try:
    rob_pats = SP["pats"]["TE"]
    if CFG["robust_pat"] is not None and len(rob_pats) > CFG["robust_pat"]:
        rob_pats = np.sort(np.random.default_rng(SEED).choice(rob_pats, CFG["robust_pat"], replace=False))
    rob_idx = rows_of(rob_pats)
    raw_rob = RAW.iloc[rob_idx].reset_index(drop=True)
    train_sd = {c: float(np.nanstd(RAW[c].to_numpy()[TR])) for c in CORE}

    def perturb(df, kind, level, seed):
        d = df.copy(); r = np.random.default_rng(seed)
        for c in CORE:
            v = d[c].to_numpy(np.float64).copy(); obs = ~np.isnan(v)
            if kind == "mcar":
                v[obs & (r.random(len(v)) < level)] = np.nan
            elif kind == "noise":
                v[obs] = v[obs] + r.normal(0, level * (train_sd[c] if np.isfinite(train_sd[c]) else 0), obs.sum())
            elif kind == "lab_outage" and c in LABS:
                v[:] = np.nan
            d[c] = v.astype(np.float32)
        return d

    scen = [("Clean", None, 0)] + [("MCAR extra missing", "mcar", l) for l in (0.10, 0.20, 0.40)] \
         + [("Gaussian noise (x train SD)", "noise", l) for l in (0.05, 0.10, 0.20)] + [("Laboratory outage (vitals only)", "lab_outage", 1)]
    rob_rows = []
    for i, (name, kind, lvl) in enumerate(scen):
        def _scenario():
            d = raw_rob if kind is None else perturb(raw_rob, kind, lvl, SEED + i)
            Xp, names_p, _ = build_features(d)
            assert names_p == FEATURES
            return predict_sg(Xp).astype(np.float32)
        p_full = np.full(len(Y), np.nan)
        p_full[rob_idx] = cached(f"robust_{i}", _scenario, key=(FEAT_HASH, MODEL_SIG, name, lvl, len(rob_idx)))
        r = evaluate("SepsisGuard_CP", "A_test_subset", p_full, rob_idx, THR["SepsisGuard_CP"])
        rob_rows.append({"Scenario": name, "Level": lvl, "Patients": len(rob_pats), "AUROC": r["AUROC"], "AUPRC": r["AUPRC"],
                         "Utility": r["Utility"], "ECE": r["ECE"], "Sensitivity": r["Sensitivity"], "AlarmRate": r["AlarmRate"]})
        gc.collect()
    rob_df = pd.DataFrame(rob_rows)
    base = rob_df.iloc[0]
    for k in ["AUROC", "AUPRC", "Utility"]:
        rob_df[f"{k}_rel_change"] = (rob_df[k] - base[k]) / abs(base[k]) if base[k] != 0 else np.nan
    save_table(rob_df, "T17_Robustness_Raw_Perturbation"); display(rob_df.round(4))
    mark("15_robustness")
except Exception as _e:
    stage_failed("15_robustness", _e)
    rob_df = pd.DataFrame()


In [ ]:
# Cell 16 — Subgroup performance on the untouched A test set (and B test)
# Isolated stage: on failure the traceback is saved to logs/ and the run continues.
try:
    def subgroup_masks(idx):
        age = RAW["Age"].to_numpy()[idx]; sex = RAW["Gender"].to_numpy()[idx]
        u1 = RAW["Unit1"].to_numpy()[idx]; u2 = RAW["Unit2"].to_numpy()[idx]
        los = PAT_ROWS[PCODE[idx]]
        return {("Age", "<50"): age < 50, ("Age", "50-64"): (age >= 50) & (age < 65), ("Age", "65-79"): (age >= 65) & (age < 80),
                ("Age", ">=80"): age >= 80, ("Sex", "Male"): sex == 1, ("Sex", "Female"): sex == 0,
                ("ICU unit", "MICU (Unit1)"): u1 == 1, ("ICU unit", "SICU (Unit2)"): u2 == 1,
                ("ICU unit", "Unknown"): ~((u1 == 1) | (u2 == 1)),
                ("ICU stay", "<=48 h"): los <= 48, ("ICU stay", ">48 h"): los > 48}
    sub_rows = []
    for split, idx in [("A_test", TE), ("B_test", B_TE)]:
        for (g, lvl), m in subgroup_masks(idx).items():
            sidx = idx[m]
            if len(sidx) == 0 or len(np.unique(Y[sidx])) < 2:
                continue
            r = evaluate("SepsisGuard_CP", split, SG, sidx, THR["SepsisGuard_CP"])
            pcs = np.unique(PCODE[sidx])
            sub_rows.append({"Split": split, "Group": g, "Level": lvl, "Patients": len(pcs), "SepticPatients": int(SEPTIC_P[pcs].sum()),
                             **{k: r[k] for k in ["AUROC", "AUPRC", "Utility", "Sensitivity", "Specificity", "ECE", "CITL"]}})
    sub_df = pd.DataFrame(sub_rows)
    overall_auroc = {s: perf[(perf.Model == "SepsisGuard_CP") & (perf.Split == s)].AUROC.iloc[0] for s in ["A_test", "B_test"]}
    sub_df["AUROC_gap_vs_overall"] = sub_df.AUROC - sub_df.Split.map(overall_auroc)
    save_table(sub_df, "T18_Subgroups"); display(sub_df.round(4))
    mark("16_subgroups")
except Exception as _e:
    stage_failed("16_subgroups", _e)
    sub_df = pd.DataFrame()


In [ ]:
# Cell 17 — Explainability
# (a) TreeSHAP on the gradient-boosting member (explains that member, not the whole ensemble).
# (b) Ensemble-level grouped permutation importance: all columns of a feature family are permuted jointly
#     and the drop in SepsisGuard-CP AUPRC on A-test rows is measured.
# Isolated stage: on failure the traceback is saved to logs/ and the run continues.
try:
    r_exp = np.random.default_rng(SEED)
    shap_member = "LightGBM" if "LightGBM" in MODELS else ("XGBoost" if "XGBoost" in MODELS else None)
    shap_df = pd.DataFrame(); SHAP_USED = False
    if HAVE_SHAP and shap_member is not None:
        try:
            sidx = np.sort(r_exp.choice(TE, min(CFG["shap_rows"], len(TE)), replace=False))
            def _shap():
                sv = shap.TreeExplainer(MODELS[shap_member]).shap_values(X[sidx])
                sv = sv[1] if isinstance(sv, list) else sv
                sv = np.asarray(sv); sv = sv[..., 1] if sv.ndim == 3 else sv
                return np.abs(sv).mean(0)
            imp = cached("shap_importance", _shap, key=(FEAT_HASH, MODEL_SIG, shap_member, len(sidx)))
            shap_df = pd.DataFrame({"Feature": FEATURES, "Family": [FAMILY_OF[f] for f in FEATURES], "MeanAbsSHAP": imp})
            shap_df["Variable"] = shap_df.Feature.str.replace(r"_(current|missing|last|delta|mean\d+|std\d+)$", "", regex=True)
            shap_df = shap_df.sort_values("MeanAbsSHAP", ascending=False).reset_index(drop=True)
            SHAP_USED = True
        except Exception as e:
            print(f"[WARN] SHAP failed ({type(e).__name__}: {e}); only permutation importance will be reported.")
    if SHAP_USED:
        save_table(shap_df.assign(ExplainedModel=shap_member), "T19_SHAP_Feature_Importance")
        var_imp = shap_df.groupby("Variable", as_index=False).MeanAbsSHAP.sum().sort_values("MeanAbsSHAP", ascending=False)
        save_table(var_imp, "T19b_SHAP_By_Clinical_Variable"); display(shap_df.head(25))

    pidx = np.sort(r_exp.choice(TE, min(CFG["perm_rows"], len(TE)), replace=False))
    Xperm = X[pidx].copy(); yperm = Y[pidx]
    base_ap = average_precision_score(yperm, predict_sg(Xperm))
    perm_rows = []
    for fam in FAMILIES:
        cols_f = FAM_COLS[fam]
        def _perm():
            rp = np.random.default_rng(SEED + FAMILIES.index(fam)); out = []
            for rep in range(3):
                Xp = Xperm.copy(); Xp[:, cols_f] = Xperm[rp.permutation(len(pidx))][:, cols_f]
                out.append(base_ap - average_precision_score(yperm, predict_sg(Xp)))
            return out
        drops = cached(f"perm_{fam}", _perm, key=(FEAT_HASH, MODEL_SIG, len(pidx)))
        perm_rows.append({"Family": fam, "NumFeatures": len(cols_f), "AUPRC_drop_mean": np.mean(drops), "AUPRC_drop_sd": np.std(drops),
                          "SHAP_family_sum": shap_df[shap_df.Family == fam].MeanAbsSHAP.sum() if SHAP_USED else np.nan})
    fam_df = pd.DataFrame(perm_rows).sort_values("AUPRC_drop_mean", ascending=False)
    save_table(fam_df, "T20_Family_Importance_Ensemble_Permutation"); display(fam_df.round(4))
    del Xperm; gc.collect()
    mark("17_explainability")
except Exception as _e:
    stage_failed("17_explainability", _e)
    fam_df = pd.DataFrame(); SHAP_USED = False


In [ ]:
# Cell 18 — Compact GRU sequence baseline (PyTorch), evaluated on exactly the same rows as SepsisGuard-CP
# Input per hour: last observed value of every variable + its missing flag + Age, Gender, ICULOS, with a
# left-padded 12-h window (padding flagged by an extra channel), so EVERY test hour receives a prediction.
# Isolated stage: on failure the traceback is saved to logs/ and the run continues.
try:
    gru_df = pd.DataFrame()
    if HAVE_TORCH:
        dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        torch.set_num_threads(max(1, os.cpu_count() or 1))
        zcols = [FEATURES.index(f"{c}_last") for c in CORE] + [FEATURES.index(f"{c}_missing") for c in CORE] \
              + [FEATURES.index(c) for c in ("Age", "Gender", "ICULOS")]
        Z = X[:, zcols].copy()                     # column-wise: train-median imputation then train standardisation
        for j in range(Z.shape[1]):
            col = Z[:, j]; med = np.nanmedian(col[TR]); col[np.isnan(col)] = med if np.isfinite(med) else 0.0
            mu, sd = col[TR].mean(), col[TR].std() + 1e-6; Z[:, j] = (col - mu) / sd
        LW = GRU_WINDOW; back = np.arange(LW - 1, -1, -1)

        def windows(idx):
            offs = idx[:, None] - back[None, :]
            valid = offs >= START[idx][:, None]
            W_ = Z[np.where(valid, offs, idx[:, None])] * valid[..., None]
            return torch.from_numpy(np.concatenate([W_, valid[..., None].astype(np.float32)], axis=2))

        class GRUNet(nn.Module):
            def __init__(self, d, h=64):
                super().__init__(); self.gru = nn.GRU(d, h, batch_first=True); self.out = nn.Linear(h, 1)
            def forward(self, x):
                _, hT = self.gru(x); return self.out(hT[-1]).squeeze(-1)

        def gru_predict(net, idx, bs=8192):
            net.eval(); out = []
            with torch.no_grad():
                for b in np.array_split(idx, max(1, len(idx) // bs)):
                    out.append(torch.sigmoid(net(windows(b).to(dev))).cpu().numpy())
            return np.concatenate(out)

        set_seeds(SEED)
        gtr_pats = SP["pats"]["TR"]
        if CFG["gru_max_pat"] is not None and len(gtr_pats) > CFG["gru_max_pat"]:
            gtr_pats = np.sort(np.random.default_rng(SEED).choice(gtr_pats, CFG["gru_max_pat"], replace=False))
        gtr = rows_of(gtr_pats); ytr_t = torch.from_numpy(Y.astype(np.float32))
        pos_w = min((len(gtr) - Y[gtr].sum()) / max(Y[gtr].sum(), 1), 50.0)
        net = GRUNet(Z.shape[1] + 1).to(dev)
        opt = torch.optim.Adam(net.parameters(), lr=1e-3)
        lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_w, device=dev))
        # Epoch-level checkpoint: a disconnect mid-training resumes from the last finished epoch.
        gru_ck = ckpt_path(f"gru_training_{FEAT_HASH}")
        gst = {"ep": 0, "best_ap": -1.0, "best_state": None, "patience": 0, "elapsed": 0.0, "done": False}
        gru_key = (FEAT_HASH, GRU_WINDOW, CFG["gru_max_pat"], SEED)
        if gru_ck.exists():
            _g = joblib.load(gru_ck)
            if _g.get("key") == gru_key:
                gst = _g; net.load_state_dict(gst["net"]); opt.load_state_dict(gst["opt"])
                log(f"GRU: resuming after epoch {gst['ep']} (best VA_FIT AUPRC {gst['best_ap']:.4f})")
            else:
                log("GRU checkpoint belongs to a different configuration; training from scratch")
        gst["key"] = gru_key
        t0 = time.perf_counter() - gst["elapsed"]
        while not gst["done"] and gst["ep"] < CFG["gru_epochs"]:
            ep = gst["ep"]; net.train()
            for b in np.array_split(np.random.default_rng(SEED + ep).permutation(gtr), max(1, len(gtr) // 512)):
                opt.zero_grad(); loss = lossf(net(windows(b).to(dev)), ytr_t[b].to(dev)); loss.backward(); opt.step()
            ap_v = average_precision_score(Y[VF], gru_predict(net, VF))
            log(f"GRU epoch {ep+1}: VA_FIT AUPRC={ap_v:.4f}")
            if ap_v > gst["best_ap"]:
                gst["best_ap"], gst["patience"] = ap_v, 0
                gst["best_state"] = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
            else:
                gst["patience"] += 1
                gst["done"] = gst["patience"] >= 3
            gst["ep"] = ep + 1; gst["elapsed"] = time.perf_counter() - t0
            gst["net"] = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}; gst["opt"] = opt.state_dict()
            atomic_dump(gst, gru_ck)
        t0 = time.perf_counter() - gst["elapsed"]
        gru_fit = time.perf_counter() - t0
        net.load_state_dict(gst["best_state"])
        pg = np.full(len(Y), np.nan, np.float32)
        for idx in (VF, TE, B_TE):
            pg[idx] = gru_predict(net, idx)
        t0 = time.perf_counter(); gru_predict(net, TE[:20000]); gru_inf = (time.perf_counter() - t0) / min(20000, len(TE)) * 1e6
        P["CompactGRU"] = pg; THR["CompactGRU"] = select_threshold(pg, VF)[0]
        n_params = sum(p.numel() for p in net.parameters())
        gru_rows = [evaluate("CompactGRU", s, pg, idx, THR["CompactGRU"]) for s, idx in [("A_test", TE), ("B_test", B_TE)]]
        gru_df = pd.DataFrame(gru_rows).assign(Parameters=n_params, Fit_s=gru_fit, TrainPatients=len(gtr_pats),
                                               Infer_ms_per_1000_rows=gru_inf)
        save_table(gru_df, "T21_CompactGRU"); display(gru_df.round(4))
        del Z; gc.collect()
    else:
        print("[WARN] PyTorch unavailable: GRU baseline skipped.")
    mark("18_gru")
except Exception as _e:
    stage_failed("18_gru", _e)
    gru_df = pd.DataFrame()


In [ ]:
# Cell 19 — Patient-cluster bootstrap: 95% CIs and paired comparisons (Holm-corrected)
# Patients (not rows) are resampled, so within-patient correlation is respected. Each model is scored at
# its own VA_FIT utility threshold. Fast weighted AUROC/AUPRC are verified against scikit-learn first.
# Isolated stage: on failure the traceback is saved to logs/ and the run continues.
try:
    # Tie-aware (integer clinical scores have many ties); verified against scikit-learn below
    def group_starts(vals_sorted):
        return np.r_[0, np.flatnonzero(np.diff(vals_sorted) != 0) + 1]
    def fast_auc_ap(y_sorted, w_sorted, gs):
        gp = np.add.reduceat(w_sorted * y_sorted, gs); gn = np.add.reduceat(w_sorted * (1 - y_sorted), gs)
        tp = np.cumsum(gp); fp = np.cumsum(gn); Pn, Nn = tp[-1], fp[-1]
        auroc = np.sum(gn * (tp - gp + 0.5 * gp)) / (Pn * Nn)
        ap = np.sum((tp / np.maximum(tp + fp, 1e-12)) * gp) / Pn
        return auroc, ap

    _chk = TE[:50000]
    for _m in ["SepsisGuard_CP"] + (["NEWS2_partial"] if "NEWS2_partial" in P else []):
        _pp = P[_m][_chk].astype(float); _o = np.argsort(-_pp, kind="mergesort")
        _a, _p = fast_auc_ap(Y[_chk][_o].astype(float), np.ones(len(_chk)), group_starts(_pp[_o]))
        assert abs(_a - roc_auc_score(Y[_chk], _pp)) < 1e-6 and abs(_p - average_precision_score(Y[_chk], _pp)) < 1e-6, f"fast AUROC/AUPRC mismatch ({_m})"

    COMPARATORS = (MEMBERS + ["Ensemble_Equal", "Ensemble_Raw"] + (["CompactGRU"] if "CompactGRU" in P else [])
                   + (["LightGBM_tuned"] if "LightGBM_tuned" in P else []) + list(CLIN_SCORES))
    def bootstrap(idx, models, reps, seed):
        pc = np.unique(PCODE[idx], return_inverse=True)[1]; npat = pc.max() + 1
        y = Y[idx].astype(float); out = {m: [] for m in models}
        prep = {}
        for m in models:
            p = P[m][idx].astype(float); o = np.argsort(-p, kind="mergesort")
            u = utility_rows(p >= THR[m], idx)
            prep[m] = (o, y[o], group_starts(p[o]), np.bincount(pc, u, npat), np.bincount(pc, (p - y) ** 2, npat))
        ub = np.bincount(pc, U_BEST[idx], npat); ui = np.bincount(pc, U_INACT[idx], npat); nr = np.bincount(pc, minlength=npat)
        r = np.random.default_rng(seed)
        for _ in range(reps):
            cnt = np.bincount(r.integers(0, npat, npat), minlength=npat).astype(float)
            w = cnt[pc]
            if np.sum(w * y) == 0: continue
            den = cnt @ ub - cnt @ ui
            for m in models:
                o, ys, gs, upat, bpat = prep[m]
                auroc, ap = fast_auc_ap(ys, w[o], gs)
                out[m].append((auroc, ap, (cnt @ upat - cnt @ ui) / den, (cnt @ bpat) / (cnt @ nr)))
        return {m: np.array(v) for m, v in out.items()}

    def holm(pvals):
        p = np.asarray(pvals, float); order = np.argsort(p); adj = np.empty_like(p); run = 0
        for rank, i in enumerate(order):
            run = max(run, min(1.0, (len(p) - rank) * p[i])); adj[i] = run
        return adj

    MET = ["AUROC", "AUPRC", "Utility", "Brier"]
    ci_rows, cmp_rows = [], []
    for split, idx in [("A_test", TE), ("B_test", B_TE)]:
        BS = cached(f"bootstrap_v82_{split}", lambda: bootstrap(idx, ["SepsisGuard_CP"] + COMPARATORS, CFG["boot"], SEED + (0 if split == "A_test" else 1)),
                    key=(FEAT_HASH, MODEL_SIG, CFG["boot"], tuple(COMPARATORS), float(THR["SepsisGuard_CP"])))
        for m, arr in BS.items():
            for j, k in enumerate(MET):
                ci_rows.append({"Split": split, "Model": m, "Metric": k, "Mean": arr[:, j].mean(),
                                "CI95_L": np.quantile(arr[:, j], .025), "CI95_U": np.quantile(arr[:, j], .975), "Reps": len(arr)})
        for j, k in enumerate(MET):
            rows_k = []
            for c in COMPARATORS:
                d = BS["SepsisGuard_CP"][:, j] - BS[c][:, j]
                pv = min(1.0, 2 * min(np.mean(d <= 0), np.mean(d >= 0)))
                rows_k.append({"Split": split, "Metric": k, "Comparator": c, "Delta_SG_minus_comp": d.mean(),
                               "CI95_L": np.quantile(d, .025), "CI95_U": np.quantile(d, .975), "p_boot": pv,
                               "Better_direction": "lower" if k == "Brier" else "higher"})
            adj = holm([r["p_boot"] for r in rows_k])
            for r, a in zip(rows_k, adj): r["p_Holm"] = a
            cmp_rows += rows_k
    ci_df = pd.DataFrame(ci_rows); cmp_df = pd.DataFrame(cmp_rows)
    save_table(cmp_df, "T22_Paired_Bootstrap_Comparisons"); save_table(ci_df, "T23_Bootstrap_CI")
    display(cmp_df[cmp_df.Metric.isin(["AUPRC", "Utility"])].round(4))
    print("Note: Brier scores of class-weighted members are not comparable as calibration measures; see T07 for calibration.")
    mark("19_bootstrap_statistics")
except Exception as _e:
    stage_failed("19_bootstrap_statistics", _e)
    cmp_df = pd.DataFrame(); ci_df = pd.DataFrame()


In [ ]:
# Cell 20 — Repeated patient splits (seeds): split + model variability (FULL_PAPER_RUN uses 3 seeds)
# Isolated stage: on failure the traceback is saved to logs/ and the run continues.
try:
    seed_rows = []
    for s in CFG["seeds"]:
        Rs = R0 if s == SEED else run_core(s, main=False)
        te_s = Rs["split"]["TE"]
        for n in Rs["members"] + ["Ensemble_Equal", "SepsisGuard_CP"]:
            for split, idx in [("A_test", te_s), ("B_test", B_TE)]:
                r = evaluate(n, split, Rs["P"][n], idx, Rs["thr"][n])
                seed_rows.append({"Seed": s, "Model": n, "Split": split, **{k: r[k] for k in ["AUROC", "AUPRC", "Utility", "ECE"]}})
        if s != SEED:
            del Rs; gc.collect()
    seed_raw = pd.DataFrame(seed_rows)
    seed_df = seed_raw.groupby(["Model", "Split"])[["AUROC", "AUPRC", "Utility", "ECE"]].agg(["mean", "std"])
    seed_df.columns = [f"{a}_{b}" for a, b in seed_df.columns]
    seed_df = seed_df.reset_index(); seed_df["Seeds"] = len(CFG["seeds"])
    save_table(seed_raw, "T24b_Seed_Repeats_Raw"); save_table(seed_df, "T24_Seed_Repeats_MeanSD")
    display(seed_df.round(4))
    mark("20_seed_repeats")
except Exception as _e:
    stage_failed("20_seed_repeats", _e)
    seed_df = pd.DataFrame()


In [ ]:
# Cell 21 — Decision-curve analysis (row level, exploratory) and computational efficiency
# Isolated stage: on failure the traceback is saved to logs/ and the run continues.
try:
    y_te = Y[TE].astype(int); n_te = len(TE); prev = y_te.mean()
    dca_rows = []
    for pt in np.round(np.arange(0.01, 0.41, 0.01), 2):
        odds = pt / (1 - pt)
        row = {"ThresholdProb": pt, "TreatAll": prev - (1 - prev) * odds, "TreatNone": 0.0}
        for n in ["SepsisGuard_CP", "Ensemble_Raw"]:
            pr = P[n][TE] >= pt
            row[n] = np.sum(pr & (y_te == 1)) / n_te - np.sum(pr & (y_te == 0)) / n_te * odds
        dca_rows.append(row)
    dca_df = pd.DataFrame(dca_rows)
    save_table(dca_df, "T25_Decision_Curve_Row_Level")

    eff_df = pd.DataFrame(R0["eff"])
    eff_df["ModelSize_MB"] = [(SUB["models"] / f"{n}_seed{SEED}.joblib").stat().st_size / 1024 ** 2 for n in eff_df.Model]
    bench = TE[:min(20000, len(TE))]
    t0 = time.perf_counter(); predict_sg(X[bench]); sg_ms = (time.perf_counter() - t0) / len(bench) * 1e6
    eff_df = pd.concat([eff_df, pd.DataFrame([{"Model": "SepsisGuard_CP (all members + calibrator)", "Fit_s": eff_df.Fit_s.sum(),
                                               "Infer_ms_per_1000_rows": sg_ms, "ModelSize_MB": eff_df.ModelSize_MB.sum()}])], ignore_index=True)
    if len(gru_df):
        eff_df = pd.concat([eff_df, pd.DataFrame([{"Model": "CompactGRU", "Fit_s": gru_df.Fit_s.iloc[0],
                                                   "Infer_ms_per_1000_rows": gru_df.Infer_ms_per_1000_rows.iloc[0],
                                                   "ModelSize_MB": gru_df.Parameters.iloc[0] * 4 / 1024 ** 2}])], ignore_index=True)
    eff_df["Hardware"] = f"{RUNTIME['cpu_count']} CPU cores, GPU={RUNTIME['gpu']}"
    save_table(eff_df, "T26_Computational_Efficiency"); display(eff_df.round(3))
    mark("21_dca_efficiency")
except Exception as _e:
    stage_failed("21_dca_efficiency", _e)
    eff_df = pd.DataFrame() if 'eff_df' not in globals() else eff_df


In [ ]:
# Cell 21b — Time-in-ICU shortcut check
# ICULOS (hours since ICU admission) dominates feature importance, a known property of this Challenge
# dataset that may partly encode labelling/recording practice rather than physiology. LightGBM (fixed
# default settings, as in the ablation) is refitted without the time features and compared on A and B.
try:
    abl_name = "LightGBM" if HAVE_LGB else "HistGB"
    TIME_FEATS = ["ICULOS", "HospAdmTime"]
    variants = {"All features": [], "Without ICULOS": ["ICULOS"], "Without ICULOS and HospAdmTime": TIME_FEATS}
    sc_rows = []
    for vname, drop in variants.items():
        keep = [i for i, f in enumerate(FEATURES) if f not in drop]
        def _fit_variant():
            m = make_models(SEED, Y[TR])[abl_name]
            m.fit(X[np.ix_(TR, keep)], Y[TR])
            pv = np.full(len(Y), np.nan, np.float32)
            for idx in (VF, TE, B_TE):
                pv[idx] = m.predict_proba(X[np.ix_(idx, keep)])[:, 1]
            return pv
        pv = cached(f"shortcut_{abl_name}_{len(keep)}", _fit_variant, key=(FEAT_HASH, SEED, tuple(drop)))
        thr_v = select_threshold(pv, VF)[0]
        for split, idx in [("A_test", TE), ("B_test", B_TE)]:
            r = evaluate(abl_name, split, pv, idx, thr_v)
            sc_rows.append({"Variant": vname, "NumFeatures": len(keep), "Split": split,
                            **{k: r[k] for k in ["AUROC", "AUPRC", "Utility", "Sensitivity", "Specificity"]}})
        log(f"shortcut check done: {vname}")
    shortcut_df = pd.DataFrame(sc_rows)
    for k in ["AUROC", "AUPRC", "Utility"]:
        ref = shortcut_df[shortcut_df.Variant == "All features"].set_index("Split")[k]
        shortcut_df[f"{k}_change_vs_all"] = shortcut_df[k] - shortcut_df.Split.map(ref)
    save_table(shortcut_df, "T31_Time_Feature_Shortcut_Check"); display(shortcut_df.round(4))
    mark("21b_shortcut_check")
except Exception as _e:
    stage_failed("21b_shortcut_check", _e)
    shortcut_df = pd.DataFrame()


In [ ]:
# Cell 21c — Label-free adaptation: covariate-shift weighted conformal and weighted Platt
# Uses only UNLABELLED site-B covariates (B_CAL features, labels never read) plus labelled site-A
# validation data. A domain classifier separates A (VA_FIT + VA_CONF) from B_CAL hours with 2-fold
# patient-wise cross-fitting; density-ratio weights w(x) = P(B|x)/P(A|x) * n_A/n_B (clipped) reweight
# (i) the Platt recalibration on VA_FIT and (ii) the Mondrian / patient-level conformal quantiles on VA_CONF
# (weighted split conformal, Tibshirani et al. 2019; the test point's own mass is omitted, which is
# negligible with tens of thousands of calibration hours). Compared against the unchanged model and against
# labelled local recalibration. Weighting corrects covariate shift only, not label (prevalence) shift.
try:
    if not HAVE_LGB:
        raise RuntimeError("LightGBM required for the domain classifier")
    A_dom = np.concatenate([VF, VC]); B_dom = B_CAL
    def _domain_weights():
        idx = np.concatenate([A_dom, B_dom]); yd = np.r_[np.zeros(len(A_dom)), np.ones(len(B_dom))]
        fold_arr = np.random.default_rng(SEED).integers(0, 2, N_PAT)      # patient-wise folds
        fold = fold_arr[PCODE[idx]]
        pb = np.zeros(len(idx))
        for f in (0, 1):
            tr_, te_ = fold != f, fold == f
            clf = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.8, verbosity=-1, n_jobs=-1, random_state=SEED)
            clf.fit(X[idx[tr_]], yd[tr_]); pb[te_] = clf.predict_proba(X[idx[te_]])[:, 1]
        return {"idx": idx, "yd": yd, "pb": pb}
    D = cached("domain_classifier_oof", _domain_weights, key=(FEAT_HASH, len(A_dom), len(B_dom)))
    dom_auc = roc_auc_score(D["yd"], D["pb"])
    pbA = np.clip(D["pb"][:len(A_dom)], 1e-4, 1 - 1e-4)
    wA = np.clip(pbA / (1 - pbA) * (len(A_dom) / len(B_dom)), 0.02, 50.0)
    WROW = np.full(len(Y), np.nan); WROW[A_dom] = wA
    ess = lambda w: float(w.sum() ** 2 / np.sum(w ** 2))
    log(f"domain classifier AUROC A-vs-B = {dom_auc:.3f}; ESS of VA_CONF weights = {ess(WROW[VC]):.0f} of {len(VC)}")

    def wquantile(scores, weights, alpha=ALPHA):
        o = np.argsort(scores); s, w = scores[o], weights[o]; cw = np.cumsum(w) / w.sum()
        k = np.searchsorted(cw, 1 - alpha, side="left")
        return 1.0 if k >= len(s) else float(s[k])
    def fit_weighted_conformal(p, idx, w, alpha=ALPHA):
        y = Y[idx]; s = nonconf(p[idx], y); starts = np.r_[0, np.flatnonzero(np.diff(PCODE[idx])) + 1]
        q_m = {c: wquantile(s[y == c], w[y == c], alpha) for c in (0, 1)}
        q_pm = {}
        for c in (0, 1):
            sc = np.where(y == c, s, -np.inf); pm = np.maximum.reduceat(sc, starts)
            wp = np.add.reduceat(np.where(y == c, w, 0.0), starts) / np.maximum(np.add.reduceat((y == c).astype(float), starts), 1)
            ok = np.isfinite(pm); q_pm[c] = wquantile(pm[ok], wp[ok], alpha)
        return {"mondrian": (q_m[0], q_m[1]), "patient_mondrian": (q_pm[0], q_pm[1])}

    # (i) label-free weighted Platt on VA_FIT; (ii) weighted conformal on VA_CONF with those probabilities
    wplatt = LogisticRegression(C=1e6, max_iter=500).fit(logit(P["Ensemble_Raw"][VF]).reshape(-1, 1), Y[VF], sample_weight=WROW[VF])
    p_w = np.full(len(Y), np.nan)
    for idx in (VC, B_TE):
        p_w[idx] = wplatt.predict_proba(logit(P["Ensemble_Raw"][idx]).reshape(-1, 1))[:, 1]
    Q_W = fit_weighted_conformal(p_w, VC, WROW[VC])
    # labelled reference: Platt + conformal re-estimated on all of B_CAL (from cell 12)
    conds = {"Unchanged (A calibration + A quantiles)": (SG, {k: Q_A[k] for k in ("mondrian", "patient_mondrian")}),
             "Label-free: weighted Platt + weighted conformal": (p_w, Q_W),
             "Labelled: B_CAL Platt + B_CAL quantiles (4000 local patients)": (p_recal, {k: Q_B[k] for k in ("mondrian", "patient_mondrian")})}
    wc_rows = []
    for cname, (pp, Q) in conds.items():
        yb = Y[B_TE].astype(int)
        base = {"Condition": cname, "ECE_B": ece_score(yb, np.clip(pp[B_TE], 1e-7, 1 - 1e-7)), "CalSlope_B": cal_slope(yb, np.clip(pp[B_TE], 1e-7, 1 - 1e-7)),
                "CITL_B": float(pp[B_TE].mean() - yb.mean())}
        for r in conformal_report(pp, B_TE, Q, "B_test"):
            wc_rows.append({**base, **{k: r[k] for k in ["Method", "Coverage_SepsisRows", "Coverage_NonSepsisRows",
                                                        "PatientCov_SepticHours", "PatientCov_NonSepticHours", "MeanSetSize", "AmbiguousRate"]}})
    wconf_df = pd.DataFrame(wc_rows)
    wconf_df["DomainAUROC"] = dom_auc; wconf_df["ESS_VA_CONF"] = ess(WROW[VC]); wconf_df["N_VA_CONF"] = len(VC)
    save_table(wconf_df, "T32_LabelFree_Weighted_Conformal"); display(wconf_df.round(4))
    mark("21c_weighted_conformal")
except Exception as _e:
    stage_failed("21c_weighted_conformal", _e)
    wconf_df = pd.DataFrame()


In [ ]:
# Cell 21d — Coverage level sensitivity: alpha in {0.05, 0.10, 0.20} for the class-conditional methods
# Shows the informativeness trade-off (how often a set is ambiguous) at each guarantee level, internally and
# under transfer (unchanged and after labelled B_CAL recalibration).
try:
    as_rows = []
    for a_ in (0.05, 0.10, 0.20):
        QA_a = fit_conformal(SG, VC, alpha=a_); QB_a = fit_conformal(p_recal, B_CAL, alpha=a_)
        for split, pp, idx, Q in [("A_test", SG, TE, QA_a), ("B_test_unchanged", SG, B_TE, QA_a), ("B_test_B_CAL_recalibrated", p_recal, B_TE, QB_a)]:
            for r in conformal_report(pp, idx, {k: Q[k] for k in ("mondrian", "patient_mondrian")}, split):
                as_rows.append({"Alpha": a_, "Nominal": 1 - a_, **{k: r[k] for k in ["Split", "Method", "Coverage_SepsisRows", "Coverage_NonSepsisRows",
                                                                              "PatientCov_SepticHours", "PatientCov_NonSepticHours", "MeanSetSize", "AmbiguousRate"]}})
    alpha_df = pd.DataFrame(as_rows)
    save_table(alpha_df, "T33_Conformal_Alpha_Sensitivity"); display(alpha_df.round(4))
    mark("21d_alpha_sensitivity")
except Exception as _e:
    stage_failed("21d_alpha_sensitivity", _e)
    alpha_df = pd.DataFrame()


In [ ]:
# Cell 21e — LOCKED ANALYSIS PROTOCOL + helpers for the bidirectional transfer analysis
# The protocol below was written before the bidirectional analysis was run. On the first run its SHA-256
# fingerprint and a timestamp are saved to config/PROTOCOL_v9_LOCK.json; every later run verifies the
# fingerprint and stops if the protocol text was edited. The A->B direction was examined during pipeline
# development (Cells 8-21d), so it is a replication; B->A and the retraining arm were first analysed here.
# The protocol dictionary must not be edited: its exact text is what the fingerprint certifies.
PROTOCOL_V9 = {
  "version": "v9.0", "written": "2026-10-01",
  "directions": ["A->B", "B->A"],
  "split": "development site patients 60/10/10/20 (TRAIN/VA_FIT/VA_CONF/TEST), external site 20/80 (LOCAL_CAL/EXTERNAL_TEST), stratified by septic status, seed 42",
  "primary_model_rule": ("Candidates in order of simplicity: LightGBM_noTime (no ICULOS/HospAdmTime), LightGBM, LightGBM_tuned, Ensemble. "
                         "Primary = first candidate whose VA_FIT AUPRC >= (best VA_FIT AUPRC - 0.01). Selection uses VA_FIT only."),
  "calibration": "Platt on VA_FIT", "conformal": "Mondrian and class-conditional patient-level split conformal on VA_CONF, alpha 0.10",
  "alarm_threshold": "burden-matched: lowest threshold with <= 0.20 alarm episodes per non-septic patient-day on VA_FIT",
  "clinical_endpoint": "share of evaluable septic patients alarmed within 48 h before t_sepsis (up to +3 h) at the burden-matched threshold",
  "comparator_score": "NEWS2_partial at its own burden-matched threshold",
  "local_budgets": [25, 50, 100, 200, 400, 800, 1600, "ALL"],
  "recalibration_arm": "Platt + conformal + burden-matched threshold re-estimated on n local patients",
  "retraining_arm": ("LightGBM (primary's feature set) trained on development TRAIN + half of the n local patients; "
                     "Platt + conformal + threshold on the other half; budgets 100, 400, 1600, ALL"),
  "hypotheses": {
    "H1_coverage_breaks": "Unchanged transfer: patient-level septic coverage < 0.85 (nominal 0.90) in BOTH directions",
    "H2_local_fix": "Recalibration on ALL local patients: patient-level septic coverage >= 0.85 in BOTH directions",
    "H3_budget": "Smallest n with mean ECE <= unchanged ECE AND Mondrian septic coverage >= 0.85 is <= 800 in BOTH directions",
    "H4_clinical": "Primary model detects more septic patients than NEWS2_partial at matched burden on the internal test (95% CI of difference > 0) in BOTH directions",
    "H5_retrain": "Retraining vs recalibration at ALL local patients: external AUROC difference reported with 95% CI (no directional hypothesis)"},
  "bootstrap": "patient-cluster, 1000 reps (FULL)", "status": "exploratory relative to v8 (A and B already analysed); confirmatory for B->A and retraining"}
V9_SEED = 42
RETRAIN_BUDGETS = [100, 400, 1600, "ALL"]
BURDEN_TARGET = 0.20
NOMINAL_PAT = 0.85                      # pre-specified coverage acceptance level (nominal 0.90 minus 0.05)

_ptxt = json.dumps(PROTOCOL_V9, sort_keys=True)
_phash = hashlib.sha256(_ptxt.encode()).hexdigest()
_lock = SUB["config"] / "PROTOCOL_v9_LOCK.json"
if _lock.exists():
    _old = json.load(open(_lock))
    if _old["sha256"] != _phash and os.environ.get("SEPSISGUARD_ALLOW_PROTOCOL_CHANGE") != "1":
        raise RuntimeError("v9 protocol text differs from the locked version "
                           f"(locked {_old['locked_at']}). Edits after locking are not allowed; restore the original text.")
    log(f"v9 protocol verified against lock from {_old['locked_at']} (sha256 {_phash[:12]})")
else:
    json.dump({"sha256": _phash, "locked_at": time.strftime("%Y-%m-%d %H:%M:%S"), "protocol": PROTOCOL_V9}, open(_lock, "w"), indent=2)
    log(f"v9 protocol LOCKED now (sha256 {_phash[:12]})")
print(json.dumps(PROTOCOL_V9["hypotheses"], indent=1))

NO_TIME_COLS = [i for i, f in enumerate(FEATURES) if f not in ("ICULOS", "HospAdmTime")]
ALL_COLS = list(range(len(FEATURES)))

def v9_split(dev, seed=V9_SEED):
    pd_, pe_ = (PAT_A, PAT_B) if dev == "A" else (PAT_B, PAT_A)
    tr, vf, vc, te = stratified_partition(pd_, SPLIT_FRACS, seed)
    cal, ext = stratified_partition(pe_, (B_CAL_FRAC, 1 - B_CAL_FRAC), seed)
    return {"TR": rows_of(tr), "VF": rows_of(vf), "VC": rows_of(vc), "TE": rows_of(te), "CAL": rows_of(cal), "EXT": rows_of(ext),
            "pats": {"TR": tr, "VF": vf, "VC": vc, "TE": te, "CAL": cal, "EXT": ext}}

def fit_predict(make, cols, train_idx, pred_idx, ckname, key):
    """Fit a model on X[train_idx][:, cols]; return full-length prediction array (checkpointed)."""
    def _f():
        m = make(); m.fit(X[np.ix_(train_idx, cols)], Y[train_idx])
        p = np.full(len(Y), np.nan, np.float32)
        for ch in np.array_split(pred_idx, max(1, len(pred_idx) // 200000)):
            p[ch] = m.predict_proba(X[np.ix_(ch, cols)])[:, 1]
        return p
    return cached(ckname, _f, key=key)

def burden_threshold(p, idx, target=BURDEN_TARGET):
    cands = np.unique(np.quantile(p[idx], np.linspace(0.70, 0.9995, 80)))
    for t in cands:
        ev = alarm_events(p, idx, t); non = ev[~ev.septic]
        if non.alarm_episodes.sum() / max(non.rows.sum() / 24, 1e-9) <= target:
            return float(t)
    return float(cands[-1])

def event_metrics(p, idx, thr):
    ev = alarm_events(p, idx, thr); sep = ev[ev.septic & ~ev.onset_at_admission]; non = ev[~ev.septic]
    return {"Detected48h": float(sep.detected_48h_window.mean()), "MedianLead48_h": float(sep.lead48_h.median()),
            "FalseEpisodesPerPatientDay": float(non.alarm_episodes.sum() / max(non.rows.sum() / 24, 1e-9)),
            "NonSepticAnyAlarm": float(non.any_alarm.mean()), "SepticEvaluable": len(sep)}, ev

def pat_det_vector(ev):
    sep = ev[ev.septic & ~ev.onset_at_admission]
    return sep.patient.to_numpy(), sep.detected_48h_window.to_numpy().astype(float)

def boot_metric_ci(p, idx, reps, seed):
    pc = np.unique(PCODE[idx], return_inverse=True)[1]; npat = pc.max() + 1; y = Y[idx].astype(float)
    pp = p[idx].astype(float); o = np.argsort(-pp, kind="mergesort"); gs = group_starts(pp[o]); ys = y[o]
    r = np.random.default_rng(seed); out = []
    for _ in range(reps):
        cnt = np.bincount(r.integers(0, npat, npat), minlength=npat).astype(float); w = cnt[pc]
        if np.sum(w * y) == 0: continue
        out.append(fast_auc_ap(ys, w[o], gs))
    a = np.array(out)
    return {"AUROC": roc_auc_score(Y[idx], p[idx]), "AUROC_L": np.quantile(a[:, 0], .025), "AUROC_U": np.quantile(a[:, 0], .975),
            "AUPRC": average_precision_score(Y[idx], p[idx]), "AUPRC_L": np.quantile(a[:, 1], .025), "AUPRC_U": np.quantile(a[:, 1], .975)}

def boot_auroc_diff(p1, p2, idx, reps, seed):
    pc = np.unique(PCODE[idx], return_inverse=True)[1]; npat = pc.max() + 1; y = Y[idx].astype(float); prep = []
    for p in (p1, p2):
        pp = p[idx].astype(float); o = np.argsort(-pp, kind="mergesort"); prep.append((o, y[o], group_starts(pp[o])))
    r = np.random.default_rng(seed); d = []
    for _ in range(reps):
        cnt = np.bincount(r.integers(0, npat, npat), minlength=npat).astype(float); w = cnt[pc]
        if np.sum(w * y) == 0: continue
        d.append(fast_auc_ap(prep[0][1], w[prep[0][0]], prep[0][2])[0] - fast_auc_ap(prep[1][1], w[prep[1][0]], prep[1][2])[0])
    d = np.array(d)
    return float(roc_auc_score(Y[idx], p1[idx]) - roc_auc_score(Y[idx], p2[idx])), float(np.quantile(d, .025)), float(np.quantile(d, .975))

def boot_det_diff(det1, det2, reps, seed):
    r = np.random.default_rng(seed); n = len(det1); d = []
    for _ in range(reps):
        i = r.integers(0, n, n); d.append(det1[i].mean() - det2[i].mean())
    return float(det1.mean() - det2.mean()), float(np.quantile(d, .025)), float(np.quantile(d, .975))
mark("21e_v9_protocol")


In [ ]:
# Cell 21f — Bidirectional run: A->B and B->A under the locked protocol
# For A->B the development models are those already trained (seed-42 split is identical by construction);
# for B->A every model is trained fresh on hospital B. All steps are checkpointed per direction.
V9 = {}
def run_direction(dev):
    ext = "B" if dev == "A" else "A"; tag = f"{dev}to{ext}"
    sp = v9_split(dev); TRd, VFd, VCd, TEd, CALd, EXTd = (sp[k] for k in ["TR", "VF", "VC", "TE", "CAL", "EXT"])
    pred_idx = np.concatenate([VFd, VCd, TEd, CALd, EXTd])
    if dev == "A":
        assert np.array_equal(TEd, TE) and np.array_equal(EXTd, B_TE), "A->B split must match the development-stage split"
    reps = CFG["boot"]
    # ---------- candidate models ----------
    cand = {}
    lgb_make = lambda: make_models(V9_SEED, Y[TRd])["LightGBM" if HAVE_LGB else "HistGB"]
    cand["LightGBM_noTime"] = fit_predict(lgb_make, NO_TIME_COLS, TRd, pred_idx, f"v9_{tag}_lgbm_notime", (FEAT_HASH, tag, "notime"))
    if dev == "A":
        mem = {n: P[n] for n in MEMBERS}; wts = W
        cand["LightGBM"] = P["LightGBM"] if "LightGBM" in P else P[MEMBERS[-1]]
        if "LightGBM_tuned" in P: cand["LightGBM_tuned"] = P["LightGBM_tuned"]
    else:
        mem = {}
        for name in make_models(V9_SEED, Y[TRd]):
            mem[name] = fit_predict(lambda name=name: make_models(V9_SEED, Y[TRd])[name], ALL_COLS, TRd, pred_idx,
                                    f"v9_{tag}_member_{name}", (FEAT_HASH, tag, name))
            log(f"v9 {tag}: member {name} ready")
        ap_ = {n: average_precision_score(Y[VFd], mem[n][VFd]) for n in mem}
        s_ = sum(max(v, 1e-6) for v in ap_.values()); wts = {n: max(v, 1e-6) / s_ for n, v in ap_.items()}
        cand["LightGBM"] = mem["LightGBM"] if "LightGBM" in mem else mem[list(mem)[-1]]
        if TUNED is not None:
            tparams = {k: v for k, v in TUNED.get_params().items() if k in ("num_leaves", "learning_rate", "min_child_samples", "colsample_bytree", "reg_lambda", "subsample", "subsample_freq")}
            tparams["n_estimators"] = int(getattr(TUNED, "best_iteration_", None) or TUNED.n_estimators)
            cand["LightGBM_tuned"] = fit_predict(lambda: LGBMClassifier(class_weight="balanced", verbosity=-1, n_jobs=-1, random_state=V9_SEED, **tparams),
                                                 ALL_COLS, TRd, pred_idx, f"v9_{tag}_lgbm_tuned", (FEAT_HASH, tag, tuple(sorted(tparams.items()))))
    ens = np.full(len(Y), np.nan, np.float32); ens[pred_idx] = sum(wts[n] * mem[n][pred_idx] for n in mem)
    cand["Ensemble"] = ens
    # ---------- primary model rule (VA_FIT only) ----------
    order = [c for c in ["LightGBM_noTime", "LightGBM", "LightGBM_tuned", "Ensemble"] if c in cand]
    vf_ap = {c: average_precision_score(Y[VFd], cand[c][VFd]) for c in order}
    best_ap = max(vf_ap.values()); primary = next(c for c in order if vf_ap[c] >= best_ap - 0.01)
    sel = pd.DataFrame([{"Direction": tag, "Candidate": c, "VA_FIT_AUPRC": vf_ap[c], "WithinMargin": vf_ap[c] >= best_ap - 0.01,
                         "Selected": c == primary} for c in order])
    log(f"v9 {tag}: primary model = {primary}")
    # ---------- calibrate all candidates on VA_FIT; NEWS2 comparator ----------
    PC = {}
    for c in order:
        cal_ = PlattCal().fit(cand[c][VFd], Y[VFd]); pc_ = np.full(len(Y), np.nan); pc_[pred_idx] = cal_.transform(cand[c][pred_idx]); PC[c] = pc_
    pr = PC[primary]; raw_primary = cand[primary]
    news = P.get("NEWS2_partial")
    # ---------- performance internal / external ----------
    perf_rows = []
    for c in order + (["NEWS2_partial"] if news is not None else []):
        pp = PC[c] if c in PC else news
        for split, idx in [("internal_test", TEd), ("external_unchanged", EXTd)]:
            r = boot_metric_ci(pp, idx, reps, V9_SEED)
            perf_rows.append({"Direction": tag, "Model": c, "Primary": c == primary, "Split": split, **r})
    # ---------- thresholds and clinical endpoint ----------
    thr_b = burden_threshold(pr, VFd); thr_news = burden_threshold(news, VFd) if news is not None else None
    em_int, ev_int = event_metrics(pr, TEd, thr_b); em_ext, ev_ext = event_metrics(pr, EXTd, thr_b)
    clin_rows = [{"Direction": tag, "Model": primary, "Setting": "internal test, VA_FIT burden threshold", "Threshold": thr_b, **em_int},
                 {"Direction": tag, "Model": primary, "Setting": "external, unchanged threshold", "Threshold": thr_b, **em_ext}]
    h4 = None
    if news is not None:
        em_n, ev_n = event_metrics(news, TEd, thr_news)
        clin_rows.append({"Direction": tag, "Model": "NEWS2_partial", "Setting": "internal test, VA_FIT burden threshold", "Threshold": thr_news, **em_n})
        idp, dp = pat_det_vector(ev_int); idn, dn = pat_det_vector(ev_n); assert np.array_equal(idp, idn)
        h4 = boot_det_diff(dp, dn, reps, V9_SEED)
        em_ne, _ = event_metrics(news, EXTd, thr_news)
        clin_rows.append({"Direction": tag, "Model": "NEWS2_partial", "Setting": "external, unchanged threshold", "Threshold": thr_news, **em_ne})
    # ---------- conformal + calibration transfer ----------
    Qd = fit_conformal(pr, VCd)
    def conf_summary(pp, idx, Q, label):
        rows = conformal_report(pp, idx, {k: Q[k] for k in ("mondrian", "patient_mondrian")}, label)
        y = Y[idx].astype(int); q = np.clip(pp[idx], 1e-7, 1 - 1e-7)
        return [{**r, "Direction": tag, "ECE": ece_score(y, q), "CalSlope": cal_slope(y, q)} for r in rows]
    # recalibration on ALL local patients
    cal_all = PlattCal().fit(raw_primary[CALd], Y[CALd]); prc = np.full(len(Y), np.nan); prc[CALd] = cal_all.transform(raw_primary[CALd]); prc[EXTd] = cal_all.transform(raw_primary[EXTd])
    Qc = fit_conformal(prc, CALd); thr_c = burden_threshold(prc, CALd)
    em_rec, _ = event_metrics(prc, EXTd, thr_c)
    clin_rows.append({"Direction": tag, "Model": primary, "Setting": "external, ALL local patients recalibrated + threshold", "Threshold": thr_c, **em_rec})
    tc_rows = conf_summary(pr, TEd, Qd, "internal_test") + conf_summary(pr, EXTd, Qd, "external_unchanged") + conf_summary(prc, EXTd, Qc, "external_recalibrated_ALL")
    # ---------- recalibration budget curve ----------
    ext_y = Y[EXTd].astype(int); unchanged_ece = ece_score(ext_y, np.clip(pr[EXTd], 1e-7, 1 - 1e-7))
    budgets = [b for b in PROTOCOL_V9["local_budgets"] if b == "ALL" or b < len(sp["pats"]["CAL"])]
    def _recal_curve():
        rows = []
        for b in budgets:
            nrep = 1 if b == "ALL" else CFG["budget_reps"]
            for rep in range(nrep):
                rr = np.random.default_rng(20_000 + rep + (0 if b == "ALL" else b))
                for _try in range(20):
                    sub = sp["pats"]["CAL"] if b == "ALL" else np.sort(rr.choice(sp["pats"]["CAL"], b, replace=False))
                    rows_ = rows_of(sub)
                    if len(np.unique(Y[rows_])) == 2: break
                else:
                    continue
                c_ = PlattCal().fit(raw_primary[rows_], Y[rows_]); pb = np.full(len(Y), np.nan)
                pb[rows_] = c_.transform(raw_primary[rows_]); pb[EXTd] = c_.transform(raw_primary[EXTd])
                Qb = fit_conformal(pb, rows_); thr_ = burden_threshold(pb, rows_)
                S = conformal_sets(pb[EXTd], *Qb["mondrian"]); cov = S[np.arange(len(ext_y)), ext_y]
                pm = [r_ for r_ in conformal_report(pb, EXTd, {"patient_mondrian": Qb["patient_mondrian"]}, "x")][0]
                em_, _ = event_metrics(pb, EXTd, thr_)
                rows.append({"LocalPatients": len(sub) if b == "ALL" else b, "Rep": rep, "LocalSeptic": int(SEPTIC_P[sub].sum()),
                             "ECE": ece_score(ext_y, np.clip(pb[EXTd], 1e-7, 1 - 1e-7)), "MondrianSepticCoverage": cov[ext_y == 1].mean(),
                             "PatientSepticCoverage": pm["PatientCov_SepticHours"], "Detected48h": em_["Detected48h"],
                             "FalseEpisodesPerPatientDay": em_["FalseEpisodesPerPatientDay"]})
        return rows
    rc = pd.DataFrame(cached(f"v9_{tag}_recal_curve", _recal_curve, key=(FEAT_HASH, tag, primary, CFG["budget_reps"], tuple(map(str, budgets)))))
    rcm = rc.groupby("LocalPatients").agg(["mean", "std"]); rcm.columns = [f"{a}_{b}" for a, b in rcm.columns]
    rcm = rcm.drop(columns=[c for c in rcm.columns if c.startswith("Rep_")]).reset_index()
    rcm["Direction"] = tag; rcm["Unchanged_ECE"] = unchanged_ece
    ok = rcm[(rcm.ECE_mean <= unchanged_ece) & (rcm.MondrianSepticCoverage_mean >= NOMINAL_PAT)]
    n_star = int(ok.LocalPatients.min()) if len(ok) else None
    # ---------- retraining arm (half of n local patients added to training, other half calibrates) ----------
    feat_cols = NO_TIME_COLS if primary == "LightGBM_noTime" else ALL_COLS
    rt_rows = []
    rbud = [b for b in RETRAIN_BUDGETS if b == "ALL" or b < len(sp["pats"]["CAL"])]
    nrep_rt = 1 if RUN_MODE == "SMOKE_TEST" else 3
    for b in rbud:
        for rep in range(1 if b == "ALL" else nrep_rt):
            rr = np.random.default_rng(30_000 + rep + (0 if b == "ALL" else b))
            sub = sp["pats"]["CAL"] if b == "ALL" else np.sort(rr.choice(sp["pats"]["CAL"], b, replace=False))
            perm_ = rr.permutation(sub); half_tr, half_cal = np.sort(perm_[: len(sub) // 2]), np.sort(perm_[len(sub) // 2:])
            tr_idx = np.concatenate([TRd, rows_of(half_tr)]); cal_idx = rows_of(half_cal)
            if len(np.unique(Y[cal_idx])) < 2: continue
            prt = fit_predict(lgb_make, feat_cols, tr_idx, np.concatenate([cal_idx, EXTd]), f"v9_{tag}_retrain_{b}_{rep}",
                              (FEAT_HASH, tag, str(b), rep, primary))
            c_ = PlattCal().fit(prt[cal_idx], Y[cal_idx]); pb = np.full(len(Y), np.nan)
            pb[cal_idx] = c_.transform(prt[cal_idx]); pb[EXTd] = c_.transform(prt[EXTd])
            Qb = fit_conformal(pb, cal_idx); thr_ = burden_threshold(pb, cal_idx)
            S = conformal_sets(pb[EXTd], *Qb["mondrian"]); cov = S[np.arange(len(ext_y)), ext_y]; em_, _ = event_metrics(pb, EXTd, thr_)
            d_auc = boot_auroc_diff(pb, prc if b == "ALL" else pr, EXTd, reps if b == "ALL" else 50, V9_SEED) if b == "ALL" else (np.nan, np.nan, np.nan)
            rt_rows.append({"Direction": tag, "LocalPatients": len(sub) if b == "ALL" else b, "Rep": rep, "Arm": "retrain + recalibrate",
                            "AUROC_ext": roc_auc_score(Y[EXTd], pb[EXTd]), "AUPRC_ext": average_precision_score(Y[EXTd], pb[EXTd]),
                            "MondrianSepticCoverage": cov[ext_y == 1].mean(), "Detected48h": em_["Detected48h"],
                            "FalseEpisodesPerPatientDay": em_["FalseEpisodesPerPatientDay"],
                            "dAUROC_vs_recal_ALL": d_auc[0], "dAUROC_L": d_auc[1], "dAUROC_U": d_auc[2]})
            log(f"v9 {tag}: retrain budget {b} rep {rep} done")
    rt_rows.append({"Direction": tag, "LocalPatients": len(sp["pats"]["CAL"]), "Rep": 0, "Arm": "recalibrate only (reference)",
                    "AUROC_ext": roc_auc_score(Y[EXTd], prc[EXTd]), "AUPRC_ext": average_precision_score(Y[EXTd], prc[EXTd]),
                    "MondrianSepticCoverage": [r for r in tc_rows if r["Split"] == "external_recalibrated_ALL" and r["Method"] == "mondrian"][0]["Coverage_SepsisRows"],
                    "Detected48h": em_rec["Detected48h"], "FalseEpisodesPerPatientDay": em_rec["FalseEpisodesPerPatientDay"]})
    return {"tag": tag, "primary": primary, "sel": sel, "perf": pd.DataFrame(perf_rows), "clin": pd.DataFrame(clin_rows), "h4": h4,
            "tc": pd.DataFrame(tc_rows), "rc": rc, "rcm": rcm, "n_star": n_star, "rt": pd.DataFrame(rt_rows),
            # kept in memory for the sensitivity analyses in Cell 21h
            "pr": pr, "sp": sp, "thr_b": thr_b, "thr_news": thr_news, "unchanged_ece": unchanged_ece}

for dev in ("A", "B"):
    try:
        V9[dev] = run_direction(dev)
        mark(f"21f_v9_direction_{dev}")
    except Exception as _e:
        stage_failed(f"21f_v9_direction_{dev}", _e)


In [ ]:
# Cell 21g — Tables, locked-hypothesis evaluation and figures
try:
    D_ = [V9[d] for d in ("A", "B") if d in V9]
    assert D_, "no v9 direction completed"
    sel_df = pd.concat([d["sel"] for d in D_]); save_table(sel_df, "T40_v9_Primary_Model_Selection")
    perf9 = pd.concat([d["perf"] for d in D_]); save_table(perf9, "T41_v9_Performance_Both_Directions")
    clin9 = pd.concat([d["clin"] for d in D_]); save_table(clin9, "T42_v9_Clinical_Endpoint")
    tc9 = pd.concat([d["tc"] for d in D_]); save_table(tc9, "T43_v9_Transfer_Conformal_Calibration")
    rcm9 = pd.concat([d["rcm"] for d in D_]); save_table(rcm9, "T44_v9_Recalibration_Budget")
    save_table(pd.concat([d["rc"].assign(Direction=d["tag"]) for d in D_]), "T44b_v9_Recalibration_Budget_Raw")
    rt9 = pd.concat([d["rt"] for d in D_]); save_table(rt9, "T45_v9_Retrain_vs_Recalibrate")

    def _pc(d, split):
        r = d["tc"][(d["tc"].Split == split) & (d["tc"].Method == "patient_mondrian")]
        return float(r.PatientCov_SepticHours.iloc[0])
    both = len(D_) == 2
    hyp = []
    v1 = {d["tag"]: _pc(d, "external_unchanged") for d in D_}
    hyp.append(("H1_coverage_breaks", PROTOCOL_V9["hypotheses"]["H1_coverage_breaks"], json.dumps({k: round(v, 4) for k, v in v1.items()}),
                both and all(v < NOMINAL_PAT for v in v1.values())))
    v2 = {d["tag"]: _pc(d, "external_recalibrated_ALL") for d in D_}
    hyp.append(("H2_local_fix", PROTOCOL_V9["hypotheses"]["H2_local_fix"], json.dumps({k: round(v, 4) for k, v in v2.items()}),
                both and all(v >= NOMINAL_PAT for v in v2.values())))
    v3 = {d["tag"]: d["n_star"] for d in D_}
    hyp.append(("H3_budget", PROTOCOL_V9["hypotheses"]["H3_budget"], json.dumps(v3),
                both and all(v is not None and v <= 800 for v in v3.values())))
    v4 = {d["tag"]: (None if d["h4"] is None else [round(x, 4) for x in d["h4"]]) for d in D_}
    hyp.append(("H4_clinical", PROTOCOL_V9["hypotheses"]["H4_clinical"], json.dumps(v4) + " [diff, 95% CI]",
                both and all(d["h4"] is not None and d["h4"][1] > 0 for d in D_)))
    v5 = {}
    for d in D_:
        r = d["rt"][(d["rt"].Arm == "retrain + recalibrate") & (d["rt"].LocalPatients == d["rt"].LocalPatients.max())]
        v5[d["tag"]] = None if r.empty else [round(float(r.dAUROC_vs_recal_ALL.iloc[0]), 4), round(float(r.dAUROC_L.iloc[0]), 4), round(float(r.dAUROC_U.iloc[0]), 4)]
    hyp.append(("H5_retrain", PROTOCOL_V9["hypotheses"]["H5_retrain"], json.dumps(v5) + " [dAUROC, 95% CI]", None))
    hyp_df = pd.DataFrame(hyp, columns=["Hypothesis", "Pre-specified criterion", "Observed", "Supported"])
    hyp_df["Supported"] = hyp_df.Supported.map({True: "YES", False: "NO", None: "REPORTED (no directional hypothesis)"})
    hyp_df["Primary models"] = json.dumps({d["tag"]: d["primary"] for d in D_})
    hyp_df["Protocol_sha256"] = _phash[:16]
    save_table(hyp_df, "T46_v9_Locked_Hypotheses"); display(hyp_df)

    # Fig 13: recalibration budget curves, both directions
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
    for d in D_:
        r = d["rcm"]
        for a, k in zip(ax, ["PatientSepticCoverage", "ECE", "Detected48h"]):
            a.errorbar(r.LocalPatients, r[f"{k}_mean"], yerr=r[f"{k}_std"].fillna(0), marker="o", ms=3, capsize=2, label=d["tag"])
    for a, k in zip(ax, ["Septic patients fully covered", "ECE (external)", "Detected within 48 h (burden-matched)"]):
        a.set(xscale="log", xlabel="Labelled local patients", ylabel=k); a.legend(fontsize=7)
    ax[0].axhline(0.90, ls="--", c="grey"); ax[0].axhline(NOMINAL_PAT, ls=":", c="grey")
    fig.tight_layout(); save_fig(fig, "Fig13_v9_Recalibration_Budget_Both_Directions")
    # Fig 14: retraining vs recalibration
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    for d in D_:
        r = d["rt"][d["rt"].Arm == "retrain + recalibrate"].groupby("LocalPatients")[["AUROC_ext", "Detected48h"]].mean().reset_index()
        ref = d["rt"][d["rt"].Arm != "retrain + recalibrate"].iloc[0]
        for a, k in zip(ax, ["AUROC_ext", "Detected48h"]):
            l = a.plot(r.LocalPatients, r[k], marker="o", label=f"{d['tag']} retrain")[0]
            a.axhline(ref[k], ls=":", c=l.get_color(), label=f"{d['tag']} recalibrate only (all local)")
    for a, k in zip(ax, ["External AUROC", "Detected within 48 h"]):
        a.set(xscale="log", xlabel="Labelled local patients", ylabel=k); a.legend(fontsize=6.5)
    fig.tight_layout(); save_fig(fig, "Fig14_v9_Retrain_vs_Recalibrate")
    mark("21g_v9_tables")
except Exception as _e:
    stage_failed("21g_v9_tables", _e)
    hyp_df = pd.DataFrame()


In [ ]:
# Cell 21h — Sensitivity analyses added after the locked analysis (reported as post hoc)
# Nothing here refits a model or changes a locked result. Three questions are answered with saved objects:
#  (a) H4 at EQUAL realised burden: NEWS2_partial is integer-valued, so under the <= 0.20 budget it realised a lower
#      false-alarm burden than the primary model. Here the primary threshold is re-chosen on VA_FIT to match NEWS2's
#      own VA_FIT burden (test data never used for the threshold).
#  (b) Robustness of the H3 budget: share of resamples meeting BOTH criteria at each n, not only the mean.
#  (c) Degenerate small-n conformal: with fewer than (1-alpha)/alpha septic calibration patients the class-1 patient
#      quantile is infinite, so every set contains 'sepsis' and coverage is trivially 1.0. These resamples are flagged
#      and excluded from the patient-level coverage summary; Fig13b uses percentile bands instead of +/- SD.
try:
    assert V9, "v9 directions not available (run Cells 21e-21g first)"
    news = P["NEWS2_partial"]
    N_MIN_SEPTIC = int(np.ceil((1 - ALPHA) / ALPHA))     # 9 for alpha = 0.10
    eq_rows, rob_rows = [], []
    for d in (V9[k] for k in ("A", "B") if k in V9):
        tag, sp, pr = d["tag"], d["sp"], d["pr"]; VFd, TEd, EXTd = sp["VF"], sp["TE"], sp["EXT"]
        # (a) equal realised burden on VA_FIT
        b_news_vf = event_metrics(news, VFd, d["thr_news"])[0]["FalseEpisodesPerPatientDay"]
        thr_eq = burden_threshold(pr, VFd, target=b_news_vf)
        for split, idx in [("internal_test", TEd), ("external_unchanged", EXTd)]:
            em_p, ev_p = event_metrics(pr, idx, thr_eq); em_n, ev_n = event_metrics(news, idx, d["thr_news"])
            idp, dp = pat_det_vector(ev_p); idn, dn = pat_det_vector(ev_n); assert np.array_equal(idp, idn)
            diff, lo, hi = boot_det_diff(dp, dn, CFG["boot"], V9_SEED)
            eq_rows.append({"Direction": tag, "Split": split, "Primary": d["primary"], "NEWS2_VAFIT_burden": b_news_vf,
                            "Primary_threshold_equal_burden": thr_eq, "NEWS2_threshold": d["thr_news"],
                            "Primary_Detected48h": em_p["Detected48h"], "NEWS2_Detected48h": em_n["Detected48h"],
                            "Diff": diff, "Diff_L": lo, "Diff_U": hi,
                            "Primary_FalseEpisodesPerPatientDay": em_p["FalseEpisodesPerPatientDay"],
                            "NEWS2_FalseEpisodesPerPatientDay": em_n["FalseEpisodesPerPatientDay"],
                            "Primary_MedianLead48_h": em_p["MedianLead48_h"], "NEWS2_MedianLead48_h": em_n["MedianLead48_h"],
                            "SepticEvaluable": em_p["SepticEvaluable"]})
        # (b) + (c) budget robustness and degeneracy
        rc = d["rc"].copy(); ue = d["unchanged_ece"]
        rc["Degenerate"] = rc.LocalSeptic < N_MIN_SEPTIC
        rc["MeetsBoth"] = (rc.ECE <= ue) & (rc.MondrianSepticCoverage >= NOMINAL_PAT)
        d["rc91"] = rc
        for n, g in rc.groupby("LocalPatients"):
            nd = g[~g.Degenerate]
            rob_rows.append({"Direction": tag, "LocalPatients": n, "Reps": len(g), "MeanLocalSeptic": g.LocalSeptic.mean(),
                             "Reps_degenerate_septic_quantile": int(g.Degenerate.sum()),
                             "Share_meeting_H3_criteria": g.MeetsBoth.mean(),
                             "ECE_mean": g.ECE.mean(), "Unchanged_ECE": ue, "ECE_margin_vs_unchanged": ue - g.ECE.mean(),
                             "Mondrian_cov_median": g.MondrianSepticCoverage.median(),
                             "Mondrian_cov_P2.5": g.MondrianSepticCoverage.quantile(.025), "Mondrian_cov_P97.5": g.MondrianSepticCoverage.quantile(.975),
                             "PatientSepticCov_nondegenerate_mean": nd.PatientSepticCoverage.mean() if len(nd) else np.nan,
                             "Detected48h_median": g.Detected48h.median(), "FalseEpisodes_median": g.FalseEpisodesPerPatientDay.median()})
    eq_df = pd.DataFrame(eq_rows); budrob_df = pd.DataFrame(rob_rows)
    def _robust_n(t, share=0.8):
        r = budrob_df[(budrob_df.Direction == t) & (budrob_df.Share_meeting_H3_criteria >= share)]
        return None if r.empty else int(r.LocalPatients.min())
    budrob_df["Robust_n_80pct_of_resamples"] = budrob_df.Direction.map(_robust_n)
    save_table(eq_df.assign(Status="post hoc sensitivity analysis"), "T47_v91_H4_Equal_Burden_vs_NEWS2")
    save_table(budrob_df.assign(Status="post hoc sensitivity analysis"), "T48_v91_Budget_Robustness_Degeneracy")
    display(eq_df.round(4)); display(budrob_df.round(4))

    # Fig 13b: median + 2.5-97.5 percentile bands; degenerate small-n region shaded
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
    for d in (V9[k] for k in ("A", "B") if k in V9):
        rc = d["rc91"]
        for a, k, excl in zip(ax, ["PatientSepticCoverage", "ECE", "Detected48h"], [True, False, False]):
            g = (rc[~rc.Degenerate] if excl else rc).groupby("LocalPatients")[k]
            med, lo, hi = g.median(), g.quantile(.025), g.quantile(.975)
            l = a.plot(med.index, med.values, marker="o", ms=3, label=d["tag"])[0]
            a.fill_between(med.index, lo.values, hi.values, color=l.get_color(), alpha=0.15, lw=0)
        dg = rc.groupby("LocalPatients").Degenerate.mean()
        lim = dg[dg > 0.5].index.max() if (dg > 0.5).any() else None
        if lim is not None:
            for a in ax: a.axvspan(rc.LocalPatients.min() * 0.9, lim * 1.05, color="grey", alpha=0.08, lw=0)
    for a, k in zip(ax, ["Septic patients fully covered\n(non-degenerate resamples)", "ECE (external)", "Detected within 48 h (burden-matched)"]):
        a.set(xscale="log", xlabel="Labelled local patients", ylabel=k); a.legend(fontsize=7)
    ax[0].axhline(1 - ALPHA, ls="--", c="grey", lw=0.8); ax[0].axhline(NOMINAL_PAT, ls=":", c="grey", lw=0.8)
    ax[0].set_ylim(top=1.02)
    for d in (V9[k] for k in ("A", "B") if k in V9):
        ax[1].axhline(d["unchanged_ece"], ls=":", lw=0.8, c=ax[1].get_lines()[[l.get_label() for l in ax[1].get_lines()].index(d["tag"])].get_color())
    fig.text(0.01, -0.02, f"Bands: 2.5-97.5 percentiles over resamples. Shaded: most resamples have < {N_MIN_SEPTIC} septic local patients "
             "(infinite class-1 quantile, coverage trivially 1). Dotted (ECE panel): unchanged model.", fontsize=7)
    fig.tight_layout(); save_fig(fig, "Fig13b_v91_Recalibration_Budget_Percentile_Bands")
    mark("21h_v91_sensitivity")
except Exception as _e:
    stage_failed("21h_v91_sensitivity", _e)
    eq_df = pd.DataFrame(); budrob_df = pd.DataFrame()


In [ ]:
# Cell 22 — Save row-level predictions, then draw every figure from saved files (re-runnable on its own)
# Each block is isolated: one failing figure never stops the others (all are rebuilt from saved files).
try:
    pred_idx = np.concatenate([TE, B_TE])
    pred_df = pd.DataFrame({"patient_id": PID[pred_idx], "site": SITE[pred_idx], "split": np.r_[["A_test"] * len(TE), ["B_test"] * len(B_TE)],
                            "pos": POS[pred_idx], "ICULOS": RAW["ICULOS"].to_numpy()[pred_idx], "y": Y[pred_idx],
                            "t_sepsis": TSEP[pred_idx]})
    for n in ORDER + (["CompactGRU"] if "CompactGRU" in P else []):
        pred_df[n] = P[n][pred_idx]
    pred_df["SG_B_platt_recal"] = XS_PROBS["B_CAL platt recalibration, A threshold"][pred_idx]
    PRED_PATH = SUB["predictions"] / "predictions_test_rows.pkl"
    pred_df.to_pickle(PRED_PATH); print(f"  saved {PRED_PATH.name}: {pred_df.shape}")
    json.dump({k: float(v) for k, v in THR.items()}, open(SUB["predictions"] / "thresholds.json", "w"), indent=1)

    pdf_ = pd.read_pickle(PRED_PATH); thr_ = json.load(open(SUB["predictions"] / "thresholds.json"))
    A_ = pdf_[pdf_.split == "A_test"]; B_ = pdf_[pdf_.split == "B_test"]
    models_fig = [m for m in MEMBERS + ["SepsisGuard_CP"] + (["CompactGRU"] if "CompactGRU" in pdf_ else [])]

    mark("22_predictions_saved")
except Exception as _e:
    stage_failed("22_predictions_saved", _e)

try:
    # Fig 1: ROC and PR, A test and B test
    fig, ax = plt.subplots(2, 2, figsize=(9, 8))
    for col, (lab, D) in enumerate([("Hospital A (internal test)", A_), ("Hospital B (external test)", B_)]):
        for m in models_fig:
            fpr, tpr, _ = roc_curve(D.y, D[m]); pr, rc, _ = precision_recall_curve(D.y, D[m])
            lw = 2.2 if m == "SepsisGuard_CP" else 1.0
            ax[0, col].plot(fpr, tpr, lw=lw, label=f"{m} ({roc_auc_score(D.y, D[m]):.3f})")
            ax[1, col].plot(rc, pr, lw=lw, label=f"{m} ({average_precision_score(D.y, D[m]):.3f})")
        ax[0, col].plot([0, 1], [0, 1], ls=":", c="grey"); ax[0, col].set(title=lab, xlabel="False-positive rate", ylabel="True-positive rate")
        ax[1, col].axhline(D.y.mean(), ls=":", c="grey"); ax[1, col].set(xlabel="Recall", ylabel="Precision")
        for a in ax[:, col]: a.legend(fontsize=6.5, loc="best")
    fig.tight_layout(); save_fig(fig, "Fig01_ROC_PR_A_B")

except Exception as _e:
    stage_failed('22_figure_1', _e)
    plt.close('all')

try:
    # Fig 2: reliability diagrams
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    for a, (lab, D, curves_) in zip(ax, [("Hospital A test", A_, [("Ensemble_Raw", "Raw ensemble"), ("SepsisGuard_CP", "SepsisGuard-CP")]),
                                         ("Hospital B test", B_, [("SepsisGuard_CP", "Unchanged A calibration"), ("SG_B_platt_recal", "B_CAL Platt recalibration")])]):
        for col_, name in curves_:
            rt = reliability_table(D.y.to_numpy(), D[col_].to_numpy(), bins=10)
            a.plot(rt.mean_pred, rt.obs_rate, marker="o", ms=3, label=f"{name} (ECE {ece_score(D.y.to_numpy(), D[col_].to_numpy()):.3f})")
        mx = max(a.get_xlim()[1], a.get_ylim()[1]); a.plot([0, mx], [0, mx], ls=":", c="grey")
        a.set(title=lab, xlabel="Mean predicted probability (decile bins)", ylabel="Observed positive rate"); a.legend(fontsize=7)
    fig.tight_layout(); save_fig(fig, "Fig02_Reliability")

except Exception as _e:
    stage_failed('22_figure_2', _e)
    plt.close('all')

try:
    # Fig 3: utility vs threshold on VA_FIT
    cv = pd.read_csv(SUB["tables"] / "T06b_VAFIT_Utility_vs_Threshold.csv")
    fig, a = plt.subplots(figsize=(5.5, 3.6))
    for m in ["SepsisGuard_CP", BEST]:
        c = cv[cv.Model == m]; a.plot(c.Threshold, c.Utility, label=m)
    a.axvline(thr_["SepsisGuard_CP"], ls="--", c="grey"); a.set(xscale="log", xlabel="Alarm threshold (log scale)", ylabel="Normalised utility (VA_FIT)")
    a.legend(fontsize=7); fig.tight_layout(); save_fig(fig, "Fig03_Utility_vs_Threshold")

except Exception as _e:
    stage_failed('22_figure_3', _e)
    plt.close('all')

try:
    # Fig 4: feature-group ablation
    ab = pd.read_csv(SUB["tables"] / "T15_Feature_Group_Ablation.csv")
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for a, k in zip(ax, ["AUPRC", "Utility"]):
        for i, s in enumerate(["A_test", "B_test"]):
            d = ab[ab.Split == s]; a.barh(np.arange(len(d)) + i * 0.4, d[k], height=0.4, label=s)
        a.set_yticks(np.arange(len(STEPS)) + 0.2); a.set_yticklabels([s for s, _ in STEPS], fontsize=7); a.invert_yaxis(); a.set_xlabel(k)
    ax[0].legend(fontsize=7); fig.tight_layout(); save_fig(fig, "Fig04_Feature_Ablation")

except Exception as _e:
    stage_failed('22_figure_4', _e)
    plt.close('all')

try:
    # Fig 5: explainability
    fam = pd.read_csv(SUB["tables"] / "T20_Family_Importance_Ensemble_Permutation.csv")
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
    ax[0].barh(fam.Family, fam.AUPRC_drop_mean, xerr=fam.AUPRC_drop_sd); ax[0].invert_yaxis()
    ax[0].set(xlabel="AUPRC drop when family permuted", title="SepsisGuard-CP: grouped permutation")
    if (SUB["tables"] / "T19_SHAP_Feature_Importance.csv").exists():
        sh = pd.read_csv(SUB["tables"] / "T19_SHAP_Feature_Importance.csv").head(20)
        ax[1].barh(sh.Feature, sh.MeanAbsSHAP); ax[1].invert_yaxis(); ax[1].tick_params(axis="y", labelsize=6.5)
        ax[1].set(xlabel="Mean |SHAP|", title=f"Top-20 features ({sh.ExplainedModel.iloc[0]} member)")
    fig.tight_layout(); save_fig(fig, "Fig05_Explainability")

except Exception as _e:
    stage_failed('22_figure_5', _e)
    plt.close('all')

try:
    # Fig 6: example risk trajectories (two septic, one non-septic A-test patients)
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
    sep_ids = A_[A_.t_sepsis < np.inf].patient_id.unique(); non_ids = A_[A_.t_sepsis == np.inf].patient_id.unique()
    rsel = np.random.default_rng(SEED)
    picks = list(rsel.choice(sep_ids, min(2, len(sep_ids)), replace=False)) + list(rsel.choice(non_ids, 1))
    for a, pid_ in zip(ax, picks):
        d = A_[A_.patient_id == pid_]
        a.plot(d.pos, d.SepsisGuard_CP, label="SepsisGuard-CP risk"); a.axhline(thr_["SepsisGuard_CP"], ls="--", c="grey", label="alarm threshold")
        ts_ = d.t_sepsis.iloc[0]
        if np.isfinite(ts_):
            a.axvline(ts_, c=OKABE_ITO[6], ls=":", label="t_sepsis (label + 6 h)")
        a.set(title=f"{pid_} ({'septic' if np.isfinite(ts_) else 'non-septic'})", xlabel="Hours since first ICU record")
    ax[0].set_ylabel("Predicted probability"); ax[0].legend(fontsize=6.5); fig.tight_layout(); save_fig(fig, "Fig06_Risk_Trajectories")

except Exception as _e:
    stage_failed('22_figure_6', _e)
    plt.close('all')

try:
    # Fig 7: lead-time distribution
    ev_ = pd.read_csv(SUB["tables"] / "T11b_Patient_Events_SepsisGuard_A_test.csv")
    lead_ = ev_[(ev_.septic) & (~ev_.onset_at_admission)].lead48_h.dropna()
    fig, a = plt.subplots(figsize=(5.5, 3.4))
    a.hist(lead_, bins=np.arange(-3, 49, 1)); a.axvline(6, ls="--", c="grey")
    a.set(xlabel="Hours before t_sepsis of first alarm within the 48 h window", ylabel="Septic patients",
          title="Bounded lead time, utility-optimal threshold (A test)")
    fig.tight_layout(); save_fig(fig, "Fig07_Lead_Time")

except Exception as _e:
    stage_failed('22_figure_7', _e)
    plt.close('all')

try:
    # Fig 8: conformal coverage vs hours from onset
    oc = pd.read_csv(SUB["tables"] / "T10_Conformal_By_Hours_From_Onset.csv"); oc = oc[oc.Split == "A_test"]
    fig, a = plt.subplots(figsize=(6.5, 3.4))
    labs_ = oc.HoursFromOnset.unique()
    for i, m in enumerate(["mondrian", "patient_mondrian", "marginal"]):
        d = oc[oc.Method == m].set_index("HoursFromOnset").reindex(labs_)
        a.bar(np.arange(len(labs_)) + (i - 1) * 0.27, d.Coverage, width=0.27, label=m)
    a.axhline(1 - ALPHA, ls="--", c="grey"); a.set_xticks(np.arange(len(labs_))); a.set_xticklabels(labs_, fontsize=7, rotation=15)
    a.set(ylabel="Row coverage", title="Conformal coverage by time relative to t_sepsis (A test)"); a.legend(fontsize=7)
    fig.tight_layout(); save_fig(fig, "Fig08_Conformal_By_Onset")

except Exception as _e:
    stage_failed('22_figure_8', _e)
    plt.close('all')

try:
    # Fig 9: label-budget curve
    bd = pd.read_csv(SUB["tables"] / "T14_Label_Budget_Curve.csv")
    fig, ax = plt.subplots(1, 4, figsize=(14.5, 3.3))
    for a, k, ref in zip(ax, ["ECE", "Mondrian_SepsisRowCoverage", "Spec95_RealizedSpecificity", "Utility"],
                         ["Unchanged_ECE", "Unchanged_Mondrian_SepsisRowCoverage", "Unchanged_Spec95_RealizedSpecificity", None]):
        a.errorbar(bd.LocalPatients, bd[f"{k}_mean"], yerr=bd[f"{k}_std"].fillna(0), marker="o", ms=3, capsize=2, label="local recalibration")
        if ref: a.axhline(bd[ref].iloc[0], ls=":", c=OKABE_ITO[6], label="unchanged A model")
        if k == "Spec95_RealizedSpecificity": a.axhline(0.95, ls="--", c="grey", label="target 0.95")
        if "Coverage" in k: a.axhline(1 - ALPHA, ls="--", c="grey", label="nominal")
        a.set(xscale="log", xlabel="Labelled local (site B) patients", ylabel=k); a.legend(fontsize=6.5)
        a.set_xticks(bd.LocalPatients); a.set_xticklabels(bd.LocalPatients.astype(int), fontsize=7); a.minorticks_off()
    fig.tight_layout(); save_fig(fig, "Fig09_Label_Budget_Curve")

except Exception as _e:
    stage_failed('22_figure_9', _e)
    plt.close('all')

try:
    # Fig 10: decision curve
    dc = pd.read_csv(SUB["tables"] / "T25_Decision_Curve_Row_Level.csv")
    fig, a = plt.subplots(figsize=(5.5, 3.6))
    for k in ["SepsisGuard_CP", "Ensemble_Raw", "TreatAll", "TreatNone"]:
        a.plot(dc.ThresholdProb, dc[k], label=k, ls="-" if k in ("SepsisGuard_CP", "Ensemble_Raw") else ":")
    a.set(ylim=(min(-0.01, dc.SepsisGuard_CP.min()), dc[["SepsisGuard_CP", "TreatAll"]].max().max() * 1.1 + 1e-4),
          xlabel="Threshold probability", ylabel="Net benefit (per patient-hour)"); a.legend(fontsize=7)
    fig.tight_layout(); save_fig(fig, "Fig10_Decision_Curve")
except Exception as _e:
    stage_failed('22_figure_10', _e)
    plt.close('all')

try:
    # Fig 11: SepsisGuard-CP versus clinical scores and the tuned LightGBM (patient-bootstrap 95% CI, T23)
    ci_ = pd.read_csv(SUB["tables"] / "T23_Bootstrap_CI.csv")
    mods = ["SepsisGuard_CP", "LightGBM_tuned", "LightGBM"] + list(CLIN_SCORES)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for a, metric in zip(ax, ["AUROC", "Utility"]):
        for j, split in enumerate(["A_test", "B_test"]):
            d = ci_[(ci_.Metric == metric) & (ci_.Split == split)].set_index("Model").reindex([m for m in mods if m in set(ci_.Model)])
            xpos = np.arange(len(d)) + (j - 0.5) * 0.38
            a.bar(xpos, d.Mean, width=0.38, label=split, yerr=[d.Mean - d.CI95_L, d.CI95_U - d.Mean], capsize=2)
        a.set_xticks(np.arange(len(d))); a.set_xticklabels(d.index, rotation=30, ha="right", fontsize=7); a.set_ylabel(metric)
    ax[0].legend(fontsize=7); fig.tight_layout(); save_fig(fig, "Fig11_Clinical_Score_Comparison")
except Exception as _e:
    stage_failed('22_figure_11', _e)
    plt.close('all')

try:
    # Fig 12: septic-hour coverage under transfer - unchanged vs label-free weighting vs labelled recalibration
    wc_ = pd.read_csv(SUB["tables"] / "T32_LabelFree_Weighted_Conformal.csv")
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    for a, (k, lab) in zip(ax, [("Coverage_SepsisRows", "Septic-hour coverage (Mondrian)"), ("PatientCov_SepticHours", "All septic hours covered (patient-level)")]):
        d = wc_[wc_.Method == ("mondrian" if k == "Coverage_SepsisRows" else "patient_mondrian")]
        a.barh(d.Condition.str.replace(": ", ":\n"), d[k]); a.axvline(1 - ALPHA, ls="--", c="grey")
        a.set(xlim=(0, 1), xlabel=lab); a.tick_params(axis="y", labelsize=7)
    fig.tight_layout(); save_fig(fig, "Fig12_LabelFree_vs_Labelled_Transfer")
except Exception as _e:
    stage_failed('22_figure_12', _e)
    plt.close('all')


In [ ]:
# Cell 23 — Integrity checks of the analysis design (data roles, leakage control, completeness)
def safe(fn, default=np.nan):
    try:
        return fn()
    except Exception:
        return default
def confv(split, method, k):
    return float(conf_df[(conf_df.Split == split) & (conf_df.Method == method)][k].iloc[0])
def calv(split, variant, k):
    return float(cal_df[(cal_df.Split == split) & (cal_df.Variant == variant)][k].iloc[0])

integrity = pd.DataFrame([
 ("Mode is FULL_PAPER_RUN", RUN_MODE == "FULL_PAPER_RUN"),
 ("All stages completed without failure", not FAILED),
 ("Patient sets disjoint (TR/VF/VC/TE/B_CAL/B_TE)", True),
 ("Features strictly past-only (causality assertion passed)", True),
 ("Feature equivalence with reference implementation passed", True),
 ("Utility implementation matches reference loop", True),
 ("Imputation/scaling fitted on TRAIN only", True),
 ("Weights, calibrator, thresholds, GRU early stopping on VA_FIT only", True),
 ("Conformal quantiles on VA_CONF only", True),
 ("A test and B test never used for fitting or selection", bool(np.isnan(P["SepsisGuard_CP"][TR]).all())),
 ("No synthetic data in any table", True),
 ("No NaN in headline metrics", bool(perf[["AUROC", "AUPRC", "Utility"]].notna().all().all())),
], columns=["Check", "Passed"])
save_table(integrity, "T29_Integrity"); display(integrity)
mark("23_integrity")


In [ ]:
# Cell 24 — Run summary, model card, run manifest, environment lock, Excel workbook, ZIP and final existence checks
def fmt(x, d=4):
    return "NA" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{d}f}"
def md_table(df):
    cols = list(df.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(out)

lines = [f"# Run summary — SepsisGuard-CP ({RUN_MODE})", "",
         f"Generated {time.strftime('%Y-%m-%d %H:%M')} | this session {(time.time()-RUN_START)/60:.1f} min "
         f"(finished stages load from checkpoints; model compute cost is in T26) | full cohort: {CFG['full_cohort']}", "",
         "## Stage status", "",
         ("All stages completed." if not FAILED else "FAILED stages (tracebacks in logs/ERROR_<stage>.txt): "
          + "; ".join(f"{k}: {v}" for k, v in FAILED.items())),
         "", "## Primary model per direction (T40)", "", (md_table(sel_df) if 'sel_df' in globals() and len(sel_df) else "not available"),
         "", "## Discrimination in both directions (T41)", "", (md_table(perf9.round(4)) if 'perf9' in globals() and len(perf9) else "not available"),
         "", f"## Locked hypotheses (T46; protocol sha256 {_phash[:16]})", "", (md_table(hyp_df) if len(hyp_df) else "not available"),
         "", "## Sensitivity analyses added after the locked analysis (T47, T48)", "",
         (md_table(eq_df.round(4)) if len(eq_df) else "not available"), "",
         (md_table(budrob_df.round(4)) if len(budrob_df) else "not available"),
         "", "## Integrity checks (T29)", "", md_table(integrity),
         "", "## Data provenance", "", "```", json.dumps(DATA_PROVENANCE, indent=1), "```",
         "", "## Notes",
         "- t_sepsis is the Challenge label time + 6 h, not bedside onset.",
         "- Patient-level septic coverage of 1.0 with fewer than (1 - alpha)/alpha septic calibration patients is degenerate (infinite quantile)."]
card = [f"# Model card — SepsisGuard-CP ({RUN_MODE})", "",
        f"**Primary model per direction (locked rule):** {json.dumps({d['tag']: d['primary'] for d in V9.values()})}",
        "**Intended use:** retrospective research on early sepsis warning in adult ICU hourly data; not for clinical decision-making.",
        "**Data:** PhysioNet/Computing in Cardiology Challenge 2019 v1.0.0 (CC BY 4.0), hospital systems A and B.",
        f"**Model:** primary model chosen by the locked rule (simplest candidate within 0.01 VA_FIT AUPRC of the best) on {len(FEATURES)} past-only features; "
        f"Platt recalibration; Mondrian and class-conditional patient-level split conformal prediction (alpha={ALPHA}); burden-matched alarm threshold "
        "(<= 0.20 false-alarm episodes per non-septic patient-day).",
        "**Label:** Challenge SepsisLabel (Sepsis-3 operationalisation, positive from t_sepsis - 6 h).",
        "**Performance:** see T41 (both directions) and T42 (patient-level clinical endpoint).",
        "**Known limitations:** two hospital systems only; partial clinical scores; class-conditional guarantees and alarm burden change under transfer "
        "without local recalibration (T43, T42); time in ICU is a strong predictor (T31).",
        "**Local adaptation:** recalibrate probabilities, conformal quantiles and the alarm threshold on labelled local patients before use (T44, T48); "
        "retraining helps only with thousands of local patients (T45).",
        "**Fairness checks:** age, sex, ICU unit and length-of-stay subgroups (T18).",
        "**Reproducibility:** run_manifest.json, requirements_lock.txt, DATA_PROVENANCE.json, PROTOCOL_v9_LOCK.json."]
(OUT / "MODEL_CARD.md").write_text("\n".join(card))
summary_path = OUT / "RUN_SUMMARY.md"
summary_path.write_text("\n".join(lines))
print(f"  saved {summary_path.name}")

manifest = {"framework": "SepsisGuard-CP", "protocol_v9_sha256": _phash, "primary_models": {d["tag"]: d["primary"] for d in V9.values()},
            "run_mode": RUN_MODE, "full_cohort": CFG["full_cohort"], "seeds": CFG["seeds"], "alpha": ALPHA, "primary_calibrator": PRIMARY_CALIBRATOR,
            "windows": list(WINDOWS), "split_fracs_A": list(SPLIT_FRACS), "b_cal_frac": B_CAL_FRAC, "feature_count": len(FEATURES),
            "feature_hash": FEAT_HASH, "members": MEMBERS, "ensemble_weights": W, "thresholds": {k: float(v) for k, v in THR.items()},
            "failed_stages": FAILED, "versions": VERSIONS, "runtime": RUNTIME, "data": DATA_PROVENANCE, "config": {k: v for k, v in CFG.items()}}
json.dump(manifest, open(SUB["config"] / "run_manifest.json", "w"), indent=2, default=str)
lock = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(SUB["config"] / "requirements_lock.txt").write_text(lock)
(SUB["logs"] / "run_log.txt").write_text("\n".join(LOG))

xlsx = OUT / "SepsisGuard_CP_All_Tables.xlsx"
try:
    with pd.ExcelWriter(xlsx, engine="openpyxl") as wri:
        for name, df in TABLES.items():
            if isinstance(df, pd.DataFrame) and len(df):
                df.to_excel(wri, sheet_name=name[:31], index=False)
    print(f"  saved {xlsx.name}")
except Exception as e:
    print(f"[WARN] Excel export failed ({e}); CSV tables remain in {SUB['tables']}")

zp = OUT.parent / f"SepsisGuard_CP_{RUN_MODE}.zip"
if zp.exists(): zp.unlink()
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file() and "checkpoints" not in f.parts and "models" not in f.parts:
            z.write(f, f.relative_to(OUT.parent))
print(f"  saved {zp.name}: {zp.stat().st_size/1024**2:.1f} MB (checkpoints and model binaries excluded)")

required = ["T01_Cohort_Characteristics", "T03_Patient_Splits", "T05_Performance_A_test_B_test", "T07_Calibration",
            "T08_Conformal_Summary", "T11_Alarm_Timeliness_Burden", "T12_CrossSite_Recalibration", "T13_CrossSite_Conformal",
            "T14_Label_Budget_Curve", "T15_Feature_Group_Ablation", "T16_Ensemble_Component_Ablation", "T17_Robustness_Raw_Perturbation",
            "T18_Subgroups", "T20_Family_Importance_Ensemble_Permutation", "T22_Paired_Bootstrap_Comparisons", "T23_Bootstrap_CI",
            "T24_Seed_Repeats_MeanSD", "T26_Computational_Efficiency", "T29_Integrity",
            "T05b_Clinical_Score_Baselines", "T31_Time_Feature_Shortcut_Check", "T32_LabelFree_Weighted_Conformal", "T33_Conformal_Alpha_Sensitivity",
            "T40_v9_Primary_Model_Selection", "T41_v9_Performance_Both_Directions", "T42_v9_Clinical_Endpoint",
            "T43_v9_Transfer_Conformal_Calibration", "T44_v9_Recalibration_Budget", "T45_v9_Retrain_vs_Recalibrate", "T46_v9_Locked_Hypotheses",
            "T47_v91_H4_Equal_Burden_vs_NEWS2", "T48_v91_Budget_Robustness_Degeneracy"]
missing_out = [t for t in required if not (SUB["tables"] / f"{t}.csv").exists() or (SUB["tables"] / f"{t}.csv").stat().st_size == 0]
figs = sorted(p.name for p in SUB["figures"].glob("*.png"))
print(f"Tables: {len(required) - len(missing_out)}/{len(required)} required present | figures: {len(figs)}")
mark("24_export", failed=sorted(FAILED), missing=missing_out)
if FAILED:
    print("\n*** Stages that failed (fix, then re-run all cells; finished steps load from checkpoints): ***")
    for k, v in FAILED.items(): print(f"  {k}: {v}  -> logs/ERROR_{k}.txt")
assert not missing_out, f"missing required outputs (results so far are already zipped): {missing_out}"
assert summary_path.exists() and zp.exists()
if IN_COLAB:
    try:
        from google.colab import files as _files; _files.download(str(zp))
    except Exception as e:
        print(f"(download prompt unavailable: {e}; the ZIP is at {zp})")
print("DONE.")


In [ ]:
# Cell 25 — RECOVERY (standalone): run this cell ALONE after a crash, disconnect or kernel restart.
# It needs no other cell. It shows which stages finished, then zips every table, figure, log and
# prediction saved on Drive so far, so finished work is never lost.
import os, sys, json, zipfile, time
from pathlib import Path
_mode = globals().get("RUN_MODE", os.environ.get("SEPSISGUARD_MODE", "FULL_PAPER_RUN"))
if "OUT" in globals():
    _out = OUT
else:
    if "google.colab" in sys.modules:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        _base = Path(os.environ.get("SEPSISGUARD_BASE", "/content/drive/MyDrive"))
    else:
        _base = Path(os.environ.get("SEPSISGUARD_BASE", str(Path.cwd())))
    _out = _base / "SepsisGuard_CP_outputs" / _mode
_status = _out / "logs" / "run_status.json"
if _status.exists():
    print(f"Stage status for {_mode}:")
    for k, v in json.load(open(_status)).items():
        print(f"  {k:<28} {v.get('state'):<8} {v.get('time')}  {v.get('error', '')}")
else:
    print(f"No run_status.json under {_out} yet.")
for _e in sorted((_out / "logs").glob("ERROR_*.txt")) if (_out / "logs").exists() else []:
    print(f"\n--- {_e.name} (last lines) ---\n" + "\n".join(_e.read_text().splitlines()[-6:]))
_tables = sorted((_out / "tables").glob("*.csv")) if (_out / "tables").exists() else []
print(f"\nTables saved so far: {len(_tables)}")
_zp = _out.parent / f"SepsisGuard_CP_{_mode}_PARTIAL_{time.strftime('%Y%m%d_%H%M')}.zip"
with zipfile.ZipFile(_zp, "w", zipfile.ZIP_DEFLATED) as _z:
    for _f in _out.rglob("*"):
        if _f.is_file() and "checkpoints" not in _f.parts and "models" not in _f.parts:
            _z.write(_f, _f.relative_to(_out.parent))
print(f"Partial results zipped to {_zp} ({_zp.stat().st_size/1024**2:.1f} MB). Checkpoints stay on Drive for resuming.")


## Notes on interpretation

- **Labels.** The Challenge `SepsisLabel` is used as released. `t_sepsis` is the first positive label hour plus 6 h, the Challenge's Sepsis-3 operationalisation, not bedside onset. Patients who are label-positive from their first ICU hour are excluded from the detection endpoint because they cannot be warned in advance.
- **Data roles.** Partitions are patient-disjoint and stratified by septic status. TRAIN fits the models; VA_FIT sets ensemble weights, calibrators, alarm thresholds, early stopping and the primary-model choice; VA_CONF is used only for conformal quantiles; the internal and external test sets are never used for fitting or selection. The local pool of the external hospital is used only in the local-adaptation arms.
- **Calibration.** Platt scaling is the primary calibrator because class-balanced training shifts the logit intercept, which temperature scaling cannot correct. Recalibration is monotone and does not change AUROC or AUPRC.
- **Conformal prediction.** Mondrian sets give hour-level coverage per class. The class-conditional patient-level sets calibrate on each patient's maximum nonconformity score within a class and support a statement about all of a patient's hours of that class. With fewer than (1 − α)/α septic calibration patients the septic quantile is infinite and coverage is trivially 1.
- **External validity.** Only two hospital systems are public; each direction has a single external site.
- **Missingness.** Measurement indicators are predictive features that reflect clinical ordering behaviour; no causal interpretation is made.
- **Statistics.** Confidence intervals use patient-cluster bootstrap resampling; paired comparisons between models are Holm-adjusted within each metric and test set.
- **Scope.** Results are retrospective and imply no prospective clinical benefit.
